# Main code

In [ ]:
# ============================================================
# Ablation A revised:
# Keep A0/A1/A2/A3/A3-KL/A4 comparison from the first code,
# but make IntervalGP-VAE implementation consistent with
# the second uploaded code.
#
# Key revisions:
#   1) Keep latent prior / regularizer ablation:
#        A0: Standard VAE prior
#        A1: Input-dependent Gaussian prior
#        A2: IntervalGP regularizer
#        A3: Joint mini-batch GP NLL
#        A3-KL: Joint mini-batch GP KL
#        A4: Sparse inducing GP
#   2) Use the same IntervalGP-VAE parameters as the second code:
#        GP_LENGTHSCALE = 7
#        GP_VARIANCE = 2.0
#        GP_NOISE = 1e-4
#        hidden_dim = 64
#        latent_dim = 1
#        causal_head_hidden_dim = 64
#   3) Use AdamW instead of Adam.
#   4) Use the same three-stage training framework:
#        Stage 1: joint VAE + causal head training
#        Stage 2: frozen VAE, train causal head
#        Stage 3: frozen causal head, refine VAE
#   5) Use the same inference framework:
#        latent GP posterior smoothing
#        ITE-space full GP posterior for lower/upper/mean ITE.
# ============================================================

import os
import time
import random
import itertools
import inspect
import textwrap
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from scipy.stats import norm
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")


# ============================================================
# 0. Global settings
# ============================================================
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.set_default_tensor_type("torch.FloatTensor")
torch.set_default_dtype(torch.float32)

OUTPUT_DIR = "ablation_A_intervalgpvae_consistent_with_second_code"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cpu"

GLOBAL_SEED = 420
SEED = GLOBAL_SEED

np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

# Set to a small number, e.g. 2, for debugging.
# Set to None to run all 24 combinations.
MAX_ITERATIONS = None

N_TRAIN = 1000
N_TEST = 200

BATCH_SIZE = 128
JOINT_EPOCHS = 200
HEAD_EPOCHS = 100
VAE_REFINE_EPOCHS = 50

LR_JOINT = 1e-3
LR_HEAD = 1e-3
LR_VAE_REFINE = 1e-4
WEIGHT_DECAY = 1e-5

INTERVALGPVAE_LATENT_DIM = 1
INTERVALGPVAE_HIDDEN_DIM = 64
CAUSAL_HEAD_HIDDEN_DIM = 64

# Exact GP parameters from the second uploaded code version.
GP_LENGTHSCALE = 7
GP_VARIANCE = 2.0
GP_NOISE = 1e-4

ITE_CI = 0.90
Z_VALUE_90 = 1.6448536269514722

# Use the second code's ITE sampling size.
N_MC = 100

# If full GP is too slow, set MAX_GP_POINTS = 300.
# If you want full GP over all training samples, keep None.
MAX_GP_POINTS = None

CHOSEN_VERSION = "u_aux"
# Options:
#   "u_aux"               : use auxiliary latent ua1 in outcome head
#   "z_to_y"              : use all z as z_y
#   "split_z_to_t_and_y"  : use last half of z as z_y
#   "plain_u"             : only u and t in outcome head


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)


set_seed(SEED)


# ============================================================
# 1. Utility functions
# ============================================================
def to_numpy_1d(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    return np.asarray(x).reshape(-1)


def safe_corr(x, y):
    x = to_numpy_1d(x)
    y = to_numpy_1d(y)

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 2:
        return np.nan

    if np.std(x[valid]) == 0 or np.std(y[valid]) == 0:
        return np.nan

    return float(np.corrcoef(x[valid], y[valid])[0, 1])


def pehe_np(est_ite, true_ite):
    est_ite = to_numpy_1d(est_ite)
    true_ite = to_numpy_1d(true_ite)

    valid = np.isfinite(est_ite) & np.isfinite(true_ite)

    if valid.sum() == 0:
        return np.nan

    return float(np.sqrt(np.mean((est_ite[valid] - true_ite[valid]) ** 2)))


def print_function_source(title, funcs):
    print(f"\n{title}:")

    if isinstance(funcs, list):
        for i, f in enumerate(funcs):
            print(f"  Function {i + 1}:")
            try:
                src = inspect.getsource(f).strip()
                print(textwrap.indent(src, "    "))
            except Exception:
                print("    [source not available]")
    else:
        try:
            src = inspect.getsource(funcs).strip()
            print(textwrap.indent(src, "  "))
        except Exception:
            print("  [source not available]")


def maybe_subsample_gp_reference(x, *ys, max_points=None, seed=0):
    n = x.shape[0]

    if max_points is None or n <= max_points:
        return (x, *ys)

    rng = np.random.default_rng(seed)
    idx = rng.choice(np.arange(n), size=max_points, replace=False)
    idx = np.sort(idx)

    idx_t = torch.tensor(idx, dtype=torch.long, device=x.device)

    out = [x[idx_t]]

    for y in ys:
        if torch.is_tensor(y):
            out.append(y[idx_t])
        else:
            out.append(np.asarray(y)[idx])

    return tuple(out)


# ============================================================
# 2. Synthetic data generation
# ============================================================
proxy_func_sets = [
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 2,
        lambda u, ua0, ua1: u ** 3,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.exp(-0.5 * u ** 2),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(0.5 * u),
        lambda u, ua0, ua1: torch.exp(0.3 * u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
        lambda u, ua0, ua1: torch.sigmoid(u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 2,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.exp(-0.2 * u ** 2),
        lambda u, ua0, ua1: torch.sin(2 * u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 3,
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
        lambda u, ua0, ua1: torch.sin(u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.tanh(0.5 * u),
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.exp(-0.3 * u ** 2),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: u ** 2,
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.log1p(torch.exp(u)),
        lambda u, ua0, ua1: torch.exp(0.5 * u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
    ],
]


def treat_func_linear_1(u_np: np.ndarray, ua0_np=None) -> np.ndarray:
    logits = 0.8 * u_np
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


def treat_func_linear_2(u_np: np.ndarray, ua0_np=None) -> np.ndarray:
    logits = 0.4 * u_np + 0.3
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


treatment_func_list = [
    treat_func_linear_1,
    treat_func_linear_2,
]


def outcome_func_1(
    u_np: np.ndarray,
    t_np: np.ndarray,
    ua0=None,
    ua1=None,
    add_noise=True,
) -> np.ndarray:
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    base = (
        0.5 * u_np
        + 0.2 * np.sin(u_np)
        + t_np.reshape(-1, 1) * (0.8 + 0.3 * u_np)
    )

    if add_noise:
        noise = np.random.normal(0.0, 0.1, size=base.shape)
        base = base + noise

    return base.astype(np.float32)


def outcome_func_2(
    u_np: np.ndarray,
    t_np: np.ndarray,
    ua0=None,
    ua1=None,
    add_noise=True,
) -> np.ndarray:
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    base = (
        0.5 * u_np
        + 0.3 * np.cos(u_np)
        + t_np.reshape(-1, 1) * (0.5 + 0.5 * u_np)
    )

    if add_noise:
        noise = np.random.normal(0.0, 0.1, size=base.shape)
        base = base + noise

    return base.astype(np.float32)


outcome_func_list = [
    outcome_func_1,
    outcome_func_2,
]


def call_outcome_func(
    outcome_func,
    u_np,
    t_np,
    z_np=None,
    ua0_np=None,
    ua1_np=None,
    add_noise=True,
):
    if isinstance(u_np, torch.Tensor):
        u_np = u_np.detach().cpu().numpy()
    if isinstance(t_np, torch.Tensor):
        t_np = t_np.detach().cpu().numpy()
    if ua0_np is not None and isinstance(ua0_np, torch.Tensor):
        ua0_np = ua0_np.detach().cpu().numpy()
    if ua1_np is not None and isinstance(ua1_np, torch.Tensor):
        ua1_np = ua1_np.detach().cpu().numpy()

    return outcome_func(
        u_np,
        t_np,
        ua0=ua0_np,
        ua1=ua1_np,
        add_noise=add_noise,
    )


def generate_synthetic_data_with_aux_uas(
    n=1000,
    noise_std=0.1,
    seed=0,
    num_proxies=None,
    proxy_funcs=None,
    outcome_func=None,
    treatment_func=None,
):
    set_seed(seed)

    u_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    ua0_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    ua1_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)

    u_tensor = torch.from_numpy(u_np)
    ua0_tensor = torch.from_numpy(ua0_np)
    ua1_tensor = torch.from_numpy(ua1_np)

    if proxy_funcs is None:
        proxy_funcs = proxy_func_sets[0]

    if num_proxies is None:
        num_proxies = len(proxy_funcs)

    proxy_funcs = proxy_funcs[:num_proxies]

    clean_z = torch.cat(
        [g(u_tensor, ua0_tensor, ua1_tensor) for g in proxy_funcs],
        dim=1,
    )

    eps_z = torch.randn(n, num_proxies) * noise_std
    z_tensor = clean_z + eps_z

    if treatment_func is None:
        treatment_func = treat_func_linear_1

    t_np = treatment_func(u_np, ua0_np)
    t_tensor = torch.from_numpy(t_np.astype(np.float32))

    if outcome_func is None:
        outcome_func = outcome_func_1

    y_np = outcome_func(
        u_np,
        t_np,
        ua0=ua0_np,
        ua1=ua1_np,
        add_noise=True,
    )

    y_tensor = torch.from_numpy(y_np.squeeze()).float()

    return {
        "z": z_tensor.float(),
        "t": t_tensor.float(),
        "y": y_tensor.float(),
        "u": u_tensor.squeeze().float(),
        "ua0": ua0_tensor.squeeze().float(),
        "ua1": ua1_tensor.squeeze().float(),
        "proxy_funcs": proxy_funcs,
        "outcome_func": outcome_func,
        "treatment_func": treatment_func,
    }


# ============================================================
# 3. Full GP posterior functions from second code framework
# ============================================================
def rbf_kernel(x1, x2=None, lengthscale=1.0, variance=1.0):
    if x2 is None:
        x2 = x1

    if x1.ndim == 1:
        x1 = x1.view(-1, 1)

    if x2.ndim == 1:
        x2 = x2.view(-1, 1)

    dists = torch.cdist(x1, x2).pow(2)

    return variance * torch.exp(-0.5 * dists / (lengthscale ** 2))


def full_gp_posterior_point(
    x_train,
    y_train,
    x_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
    jitter=1e-5,
):
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + noise * torch.eye(n_train, device=x_train.device)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(
                    n_train,
                    device=x_train.device,
                )
            )
            break
        except RuntimeError:
            current_jitter *= 10.0
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_point.")

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v

    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def full_gp_posterior_heteroscedastic(
    x_train,
    y_train,
    x_test,
    noise_var_train,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
    jitter=1e-5,
):
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    noise_var_train = noise_var_train.float().view(-1)
    noise_var_train = torch.clamp(noise_var_train, min=min_noise)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + torch.diag(noise_var_train)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(
                    n_train,
                    device=x_train.device,
                )
            )
            break
        except RuntimeError:
            current_jitter *= 10.0
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_heteroscedastic.")

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v

    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def latent_gp_posterior_smoothing(
    z_train,
    mu_u_train,
    z_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
):
    if mu_u_train.ndim == 1:
        mu_u_train = mu_u_train.view(-1, 1)

    latent_dim = mu_u_train.shape[1]
    means = []
    vars_diag = []

    for r in range(latent_dim):
        mean_r, cov_r = full_gp_posterior_point(
            x_train=z_train,
            y_train=mu_u_train[:, r],
            x_test=z_test,
            lengthscale=lengthscale,
            variance=variance,
            noise=noise,
        )

        var_r = torch.diag(cov_r).clamp_min(1e-8)

        means.append(mean_r)
        vars_diag.append(var_r)

    mean = torch.stack(means, dim=1)
    std = torch.sqrt(torch.stack(vars_diag, dim=1))

    return mean, std


# ============================================================
# 4. Model components
# ============================================================
class DoubleEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.mean_u = nn.Linear(hidden_dim, latent_dim)
        self.logvar_u = nn.Linear(hidden_dim, latent_dim)

        self.mean_eps = nn.Linear(hidden_dim, input_dim)
        self.logvar_eps = nn.Linear(hidden_dim, input_dim)

        if self.use_aux:
            self.mean_ua0 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua0 = nn.Linear(hidden_dim, latent_dim)

            self.mean_ua1 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua1 = nn.Linear(hidden_dim, latent_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))

        mu_u = self.mean_u(h)
        logvar_u = self.logvar_u(h)
        std_u = torch.exp(0.5 * logvar_u)

        mu_eps = self.mean_eps(h)
        logvar_eps = self.logvar_eps(h)
        std_eps = torch.exp(0.5 * logvar_eps)

        if self.use_aux:
            mu_ua0 = self.mean_ua0(h)
            logvar_ua0 = self.logvar_ua0(h)
            std_ua0 = torch.exp(0.5 * logvar_ua0)

            mu_ua1 = self.mean_ua1(h)
            logvar_ua1 = self.logvar_ua1(h)
            std_ua1 = torch.exp(0.5 * logvar_ua1)

            return (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            )

        return mu_u, std_u, logvar_u, mu_eps, std_eps, logvar_eps


class AdditiveDecoder(nn.Module):
    def __init__(
        self,
        latent_dim,
        hidden_dim,
        output_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        factor = 1 + (2 if use_auxiliary_latents else 0)

        self.fc1 = nn.Linear(latent_dim * factor, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, u, eps, ua0=None, ua1=None):
        if self.use_aux:
            if ua0 is None or ua1 is None:
                raise ValueError("ua0 and ua1 are required when use_auxiliary_latents=True.")
            x = torch.cat([u, ua0, ua1], dim=1)
        else:
            x = u

        h = F.relu(self.fc1(x))

        return self.out(h) + eps


class FlexibleCausalHead(nn.Module):
    def __init__(
        self,
        latent_dim_u,
        z_y_dim=None,
        use_auxiliary_latents=False,
        hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        self.z_y_dim = z_y_dim

        input_dim = latent_dim_u + 1

        if z_y_dim is not None:
            input_dim += z_y_dim

        if use_auxiliary_latents:
            input_dim += latent_dim_u

        self.fc = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, u, t, z_y=None, ua1=None):
        t = t.view(-1, 1)

        parts = [u, t]

        if self.z_y_dim is not None:
            if z_y is None:
                raise ValueError("z_y is required but got None.")
            parts.append(z_y)

        if self.use_aux:
            if ua1 is None:
                raise ValueError("ua1 is required but got None.")
            parts.append(ua1)

        x = torch.cat(parts, dim=1)
        h = F.relu(self.fc(x))

        return self.out(h)


# ============================================================
# 5. Ablation-ready VAE backbone, but with second-code framework
# ============================================================
class GPVAEwithNoise(nn.Module):
    """
    prior_type:
        "standard"        : A0, standard VAE prior N(0,I)
        "input_gaussian"  : A1, input-dependent Gaussian prior
        "interval_gp"     : A2, IntervalGP regularizer
        "joint_gp"        : A3, joint mini-batch GP NLL
        "joint_gp_kl"     : A3-KL, joint mini-batch GP KL
        "sparse_gp"       : A4, sparse inducing GP approximation
    """

    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        use_auxiliary_latents=False,
        prior_type="interval_gp",
        prior_weight=1.0,
        std_penalty_weight=10.0,
        inducing_points=None,
        input_prior_net_hidden=32,
        jitter=1e-5,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.encoder = DoubleEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.decoder = AdditiveDecoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            output_dim=input_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.gp_lengthscale = gp_lengthscale
        self.gp_variance = gp_variance
        self.gp_noise = gp_noise

        self.prior_type = prior_type
        self.prior_weight = prior_weight
        self.std_penalty_weight = std_penalty_weight
        self.jitter = jitter

        self.input_prior_net = nn.Sequential(
            nn.Linear(input_dim, input_prior_net_hidden),
            nn.ReLU(),
            nn.Linear(input_prior_net_hidden, latent_dim),
        )

        if inducing_points is not None:
            self.register_buffer("inducing_points", inducing_points.clone().float())
        else:
            self.inducing_points = None

    def reparameterize(self, mu, std):
        return mu + torch.randn_like(std) * std

    def rbf_kernel(self, x1, x2=None):
        if x2 is None:
            x2 = x1

        if x1.ndim == 1:
            x1 = x1.view(-1, 1)

        if x2.ndim == 1:
            x2 = x2.view(-1, 1)

        dists = torch.cdist(x1, x2).pow(2)

        return self.gp_variance * torch.exp(
            -0.5 * dists / (self.gp_lengthscale ** 2)
        )

    def stable_cholesky(self, K, max_tries=8):
        eye = torch.eye(K.shape[0], device=K.device, dtype=K.dtype)
        jitter = self.jitter

        for _ in range(max_tries):
            try:
                return torch.linalg.cholesky(K + jitter * eye)
            except RuntimeError:
                jitter *= 10.0

        raise RuntimeError("Cholesky decomposition failed.")

    # ------------------------------
    # A0
    # ------------------------------
    def standard_normal_kl(self, mu_u, logvar_u):
        return -0.5 * torch.sum(
            1.0 + logvar_u - mu_u.pow(2) - logvar_u.exp()
        )

    # ------------------------------
    # A1
    # ------------------------------
    def input_dependent_gaussian_kl(self, z, mu_u, std_u, logvar_u):
        log_s2 = self.input_prior_net(z)
        log_s2 = torch.clamp(log_s2, min=-8.0, max=8.0)
        s2 = torch.exp(log_s2)

        q_var = std_u.pow(2)

        kl = 0.5 * torch.sum(
            log_s2 - logvar_u + (q_var + mu_u.pow(2)) / s2 - 1.0
        )

        return kl

    # ------------------------------
    # A2
    # ------------------------------
    def interval_gp_regularizer(self, z, mu_u, std_u):
        """
        This keeps the A2 per-sample IntervalGP regularizer,
        but uses the GP parameters from the second code.
        Full GP posterior is applied at inference.
        """
        marginal_var = self.gp_variance + self.gp_noise
        marginal_std = torch.sqrt(
            torch.tensor(marginal_var, device=z.device, dtype=z.dtype)
        )

        lower = mu_u - std_u
        upper = mu_u + std_u

        dist = torch.distributions.Normal(
            loc=torch.zeros_like(mu_u),
            scale=marginal_std * torch.ones_like(mu_u),
        )

        prob = dist.cdf(upper) - dist.cdf(lower)
        prob = torch.clamp(prob, min=1e-8)

        return -torch.sum(torch.log(prob))

    # ------------------------------
    # A3
    # ------------------------------
    def joint_gp_nll(self, z, mu_u):
        B, d = mu_u.shape

        K = self.rbf_kernel(z, z)
        K = K + (self.gp_noise + self.jitter) * torch.eye(
            B,
            device=z.device,
            dtype=z.dtype,
        )

        L = self.stable_cholesky(K)

        loss = 0.0

        for j in range(d):
            y = mu_u[:, j:j + 1]
            alpha = torch.cholesky_solve(y, L)

            quad = 0.5 * (y.T @ alpha).squeeze()
            logdet = torch.sum(torch.log(torch.diag(L)))
            const = 0.5 * B * np.log(2.0 * np.pi)

            loss = loss + quad + logdet + const

        return loss

    # ------------------------------
    # A3-KL
    # ------------------------------
    def joint_gp_kl(self, z, mu_u, std_u):
        B, d = mu_u.shape

        K = self.rbf_kernel(z, z)
        K = K + (self.gp_noise + self.jitter) * torch.eye(
            B,
            device=z.device,
            dtype=z.dtype,
        )

        L = self.stable_cholesky(K)

        K_inv = torch.cholesky_inverse(L)
        K_inv_diag = K_inv.diag()
        logdet_K = 2.0 * torch.sum(torch.log(torch.diag(L)))

        loss = 0.0

        for j in range(d):
            mu_j = mu_u[:, j:j + 1]
            var_j = std_u[:, j].pow(2) + 1e-8

            alpha = torch.cholesky_solve(mu_j, L)
            quad = (mu_j.T @ alpha).squeeze()
            trace = torch.sum(K_inv_diag * var_j)
            logdet_q = torch.sum(torch.log(var_j))

            kl_j = 0.5 * (trace + quad - B + logdet_K - logdet_q)
            loss = loss + kl_j

        return loss

    # ------------------------------
    # A4
    # ------------------------------
    def sparse_gp_nll(self, z, mu_u):
        if self.inducing_points is None:
            raise ValueError("inducing_points must be provided for sparse_gp.")

        B, d = mu_u.shape

        Zm = self.inducing_points.to(device=z.device, dtype=z.dtype)

        K_mm = self.rbf_kernel(Zm, Zm)
        K_mm = K_mm + self.jitter * torch.eye(
            K_mm.shape[0],
            device=z.device,
            dtype=z.dtype,
        )

        L_mm = self.stable_cholesky(K_mm)

        K_bm = self.rbf_kernel(z, Zm)
        K_mb = K_bm.T

        Kmm_inv_Kmb = torch.cholesky_solve(K_mb, L_mm)
        Q_bb = K_bm @ Kmm_inv_Kmb

        K_approx = Q_bb + (self.gp_noise + self.jitter) * torch.eye(
            B,
            device=z.device,
            dtype=z.dtype,
        )

        L = self.stable_cholesky(K_approx)

        loss = 0.0

        for j in range(d):
            y = mu_u[:, j:j + 1]
            alpha = torch.cholesky_solve(y, L)

            quad = 0.5 * (y.T @ alpha).squeeze()
            logdet = torch.sum(torch.log(torch.diag(L)))
            const = 0.5 * B * np.log(2.0 * np.pi)

            loss = loss + quad + logdet + const

        return loss

    def latent_prior_regularizer(self, z, mu_u, std_u, logvar_u):
        if self.prior_type == "standard":
            return self.standard_normal_kl(mu_u, logvar_u)

        if self.prior_type == "input_gaussian":
            return self.input_dependent_gaussian_kl(z, mu_u, std_u, logvar_u)

        if self.prior_type == "interval_gp":
            return self.interval_gp_regularizer(z, mu_u, std_u)

        if self.prior_type == "joint_gp":
            return self.joint_gp_nll(z, mu_u)

        if self.prior_type == "joint_gp_kl":
            return self.joint_gp_kl(z, mu_u, std_u)

        if self.prior_type == "sparse_gp":
            return self.sparse_gp_nll(z, mu_u)

        raise ValueError(f"Unknown prior_type: {self.prior_type}")

    def get_latent_stats(self, z, var_name="u"):
        outputs = self.encoder(z)

        if var_name == "u":
            return outputs[0], outputs[1], outputs[2]

        if var_name == "eps":
            return outputs[3], outputs[4], outputs[5]

        if var_name == "ua0":
            if not self.use_aux:
                return None, None, None
            return outputs[6], outputs[7], outputs[8]

        if var_name == "ua1":
            if not self.use_aux:
                return None, None, None
            return outputs[9], outputs[10], outputs[11]

        raise ValueError(f"Invalid var_name: {var_name}")

    def sample_latent(self, z, var_name="u"):
        mu, std, _ = self.get_latent_stats(z, var_name=var_name)

        if mu is None:
            return None

        return self.reparameterize(mu, std)

    def forward(self, z):
        if self.use_aux:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = self.reparameterize(mu_ua0, std_ua0)
            ua1 = self.reparameterize(mu_ua1, std_ua1)

            z_recon = self.decoder(u, eps, ua0=ua0, ua1=ua1)

        else:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = None
            ua1 = None

            z_recon = self.decoder(u, eps)

        recon_loss = F.mse_loss(z_recon, z, reduction="sum")

        prior_loss = self.latent_prior_regularizer(
            z=z,
            mu_u=mu_u,
            std_u=std_u,
            logvar_u=logvar_u,
        )

        kl_eps = -0.5 * torch.sum(
            1.0 + logvar_eps - mu_eps.pow(2) - logvar_eps.exp()
        )

        std_penalty = torch.sum(std_u ** 2)

        total_loss = (
            recon_loss
            + self.prior_weight * prior_loss
            + kl_eps
            + self.std_penalty_weight * std_penalty
        )

        if self.use_aux:
            kl_ua0 = -0.5 * torch.sum(
                1.0 + logvar_ua0 - mu_ua0.pow(2) - logvar_ua0.exp()
            )

            kl_ua1 = -0.5 * torch.sum(
                1.0 + logvar_ua1 - mu_ua1.pow(2) - logvar_ua1.exp()
            )

            total_loss = total_loss + kl_ua0 + kl_ua1

        return total_loss, {
            "recon_loss": float(recon_loss.detach().cpu()),
            "prior_loss": float(prior_loss.detach().cpu()),
            "kl_eps": float(kl_eps.detach().cpu()),
            "std_penalty": float(std_penalty.detach().cpu()),
            "u": u,
            "eps": eps,
            "ua0": ua0,
            "ua1": ua1,
            "mu_u": mu_u,
            "std_u": std_u,
            "z_recon": z_recon,
        }


class CausalGPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        z_y_dim=None,
        use_auxiliary_latents=False,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        prior_type="interval_gp",
        prior_weight=1.0,
        std_penalty_weight=10.0,
        inducing_points=None,
    ):
        super().__init__()

        self.z_y_dim = z_y_dim
        self.use_aux = use_auxiliary_latents

        self.vae = GPVAEwithNoise(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            gp_lengthscale=gp_lengthscale,
            gp_variance=gp_variance,
            gp_noise=gp_noise,
            use_auxiliary_latents=use_auxiliary_latents,
            prior_type=prior_type,
            prior_weight=prior_weight,
            std_penalty_weight=std_penalty_weight,
            inducing_points=inducing_points,
        )

        self.causal_head = FlexibleCausalHead(
            latent_dim_u=latent_dim,
            z_y_dim=z_y_dim,
            use_auxiliary_latents=use_auxiliary_latents,
            hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
        )

    def forward(self, z, t, y):
        loss_vae, vae_info = self.vae(z)

        u = vae_info["u"]
        ua1 = vae_info["ua1"] if self.use_aux else None

        z_y = make_z_y(self, z)

        y_pred = self.causal_head(
            u,
            t,
            z_y=z_y,
            ua1=ua1,
        )

        causal_loss = F.mse_loss(
            y_pred,
            y.unsqueeze(-1),
            reduction="sum",
        )

        total_loss = loss_vae + causal_loss

        return total_loss, {
            **vae_info,
            "y_pred": y_pred,
            "causal_loss": float(causal_loss.detach().cpu()),
        }


def make_z_y(model, z):
    if model.causal_head.z_y_dim == z.shape[1]:
        return z

    if model.causal_head.z_y_dim:
        return z[:, z.shape[1] // 2:]

    return None


# ============================================================
# 6. ITE-space IntervalGP inference from second code
# ============================================================
def sample_ite_from_encoder(
    model,
    z,
    n_samples=100,
):
    model.eval()

    with torch.no_grad():
        mu_u, std_u, _ = model.vae.get_latent_stats(z, var_name="u")
        N, d_u = mu_u.shape
        device = z.device

        z_y = make_z_y(model, z)

        mu_ua1, std_ua1, _ = model.vae.get_latent_stats(z, var_name="ua1")
        has_ua1 = (mu_ua1 is not None) and (std_ua1 is not None)

        eps_u = torch.randn(n_samples, N, d_u, device=device)
        u_samples = mu_u.unsqueeze(0) + eps_u * std_u.unsqueeze(0)

        U = u_samples.reshape(-1, d_u)

        if has_ua1:
            d_ua = mu_ua1.shape[1]
            eps_ua1 = torch.randn(n_samples, N, d_ua, device=device)
            ua1_samples = mu_ua1.unsqueeze(0) + eps_ua1 * std_ua1.unsqueeze(0)
            UA1 = ua1_samples.reshape(-1, d_ua)
        else:
            UA1 = None

        if z_y is not None:
            ZY = (
                z_y.unsqueeze(0)
                .expand(n_samples, -1, -1)
                .reshape(-1, z_y.shape[1])
            )
        else:
            ZY = None

        t0 = torch.zeros(U.shape[0], device=device)
        t1 = torch.ones(U.shape[0], device=device)

        y0 = model.causal_head(U, t0, z_y=ZY, ua1=UA1).squeeze(-1)
        y1 = model.causal_head(U, t1, z_y=ZY, ua1=UA1).squeeze(-1)

        ite_samples = (y1 - y0).view(n_samples, N)

        lower_q = (1.0 - ITE_CI) / 2.0
        upper_q = 1.0 - lower_q

        ite_mean = ite_samples.mean(dim=0)
        ite_std = ite_samples.std(dim=0)
        ite_lower = ite_samples.quantile(lower_q, dim=0)
        ite_upper = ite_samples.quantile(upper_q, dim=0)

        return ite_samples, ite_mean, ite_std, ite_lower, ite_upper


def ite_space_intervalgp_head(
    model,
    z_train,
    z_test,
    n_samples=100,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
):
    model.eval()

    with torch.no_grad():
        _, ite_mean_train, ite_std_train, ite_lower_train, ite_upper_train = (
            sample_ite_from_encoder(
                model=model,
                z=z_train,
                n_samples=n_samples,
            )
        )

        mu_u_train, _, _ = model.vae.get_latent_stats(z_train, var_name="u")
        mu_u_test, _, _ = model.vae.get_latent_stats(z_test, var_name="u")

        # Latent GP posterior smoothing before ITE-space GP.
        u_test_smooth, u_test_smooth_std = latent_gp_posterior_smoothing(
            z_train=z_train,
            mu_u_train=mu_u_train,
            z_test=z_test,
            lengthscale=lengthscale,
            variance=variance,
            noise=min_noise,
        )

        u_train_smooth = mu_u_train.detach()

        (
            u_train_gp,
            ite_lower_train_gp,
            ite_upper_train_gp,
            ite_mean_train_gp,
            ite_std_train_gp,
        ) = maybe_subsample_gp_reference(
            u_train_smooth,
            ite_lower_train.detach(),
            ite_upper_train.detach(),
            ite_mean_train.detach(),
            ite_std_train.detach(),
            max_points=MAX_GP_POINTS,
            seed=123,
        )

        noise_var_lower = ite_std_train_gp.pow(2) + min_noise
        noise_var_upper = ite_std_train_gp.pow(2) + min_noise
        noise_var_mean = ite_std_train_gp.pow(2) + min_noise

        gp_lower_mean, gp_lower_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_lower_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_lower,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_upper_mean, gp_upper_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_upper_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_upper,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_ite_mean, gp_ite_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_mean_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_mean,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_lower_std = torch.sqrt(torch.diag(gp_lower_cov).clamp_min(1e-8))
        gp_upper_std = torch.sqrt(torch.diag(gp_upper_cov).clamp_min(1e-8))
        gp_ite_std = torch.sqrt(torch.diag(gp_ite_cov).clamp_min(1e-8))

        raw_lower = torch.minimum(gp_lower_mean, gp_upper_mean)
        raw_upper = torch.maximum(gp_lower_mean, gp_upper_mean)

        ite_lower_test = raw_lower - Z_VALUE_90 * gp_lower_std
        ite_upper_test = raw_upper + Z_VALUE_90 * gp_upper_std

        return {
            "ite_mean_test": gp_ite_mean,
            "ite_std_test": gp_ite_std,
            "ite_lower_test": ite_lower_test,
            "ite_upper_test": ite_upper_test,
            "u_test_smooth": u_test_smooth,
            "u_test_smooth_std": u_test_smooth_std,
            "u_test_raw": mu_u_test,
            "ite_lower_train": ite_lower_train,
            "ite_upper_train": ite_upper_train,
            "ite_mean_train": ite_mean_train,
            "ite_std_train": ite_std_train,
        }


# ============================================================
# 7. Metrics
# ============================================================
def interval_metrics(ite_mean, ite_lower, ite_upper, true_ite, nominal=0.90):
    ite_mean = np.asarray(ite_mean).reshape(-1)
    ite_lower = np.asarray(ite_lower).reshape(-1)
    ite_upper = np.asarray(ite_upper).reshape(-1)
    true_ite = np.asarray(true_ite).reshape(-1)

    pehe = float(np.sqrt(np.mean((ite_mean - true_ite) ** 2)))

    ate_est = float(np.mean(ite_mean))
    ate_true = float(np.mean(true_ite))
    ate_error = float(np.abs(ate_est - ate_true))

    covered = (ite_lower <= true_ite) & (true_ite <= ite_upper)
    coverage = float(np.mean(covered))

    width = float(np.mean(ite_upper - ite_lower))
    calibration_error = float(np.abs(coverage - nominal))

    return {
        "PEHE": pehe,
        "ATE_est": ate_est,
        "ATE_true": ate_true,
        "ATE_error": ate_error,
        "coverage": coverage,
        "interval_width": width,
        "calibration_error": calibration_error,
    }


def gaussian_nll_from_interval(ite_mean, ite_lower, ite_upper, true_ite, ci=0.90):
    z_value = norm.ppf(0.5 + ci / 2.0)

    ite_mean = np.asarray(ite_mean).reshape(-1)
    ite_lower = np.asarray(ite_lower).reshape(-1)
    ite_upper = np.asarray(ite_upper).reshape(-1)
    true_ite = np.asarray(true_ite).reshape(-1)

    std = (ite_upper - ite_lower) / (2.0 * z_value)
    std = np.maximum(std, 1e-6)

    nll = (
        0.5 * np.log(2.0 * np.pi * std ** 2)
        + 0.5 * ((true_ite - ite_mean) / std) ** 2
    )

    return float(np.mean(nll))


def interval_score(ite_lower, ite_upper, true_ite, alpha=0.10):
    lower = np.asarray(ite_lower).reshape(-1)
    upper = np.asarray(ite_upper).reshape(-1)
    y = np.asarray(true_ite).reshape(-1)

    width = upper - lower
    below = y < lower
    above = y > upper

    score = (
        width
        + (2.0 / alpha) * (lower - y) * below
        + (2.0 / alpha) * (y - upper) * above
    )

    return float(np.mean(score))


def align_recovered_u_to_true_u(u_recovered, u_true):
    u_recovered = to_numpy_1d(u_recovered)
    u_true = to_numpy_1d(u_true)

    valid = np.isfinite(u_recovered) & np.isfinite(u_true)

    if valid.sum() < 2:
        return u_recovered, np.nan, np.nan, np.nan, np.nan

    x = u_recovered[valid]
    y = u_true[valid]

    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]

    u_aligned = a * u_recovered + b

    corr = safe_corr(u_aligned, u_true)
    rmse = float(np.sqrt(np.mean((u_aligned[valid] - u_true[valid]) ** 2)))

    return u_aligned, float(a), float(b), corr, rmse


def latent_rmse_and_corr(mu_u, true_u):
    mu_np = to_numpy_1d(mu_u)
    true_np = to_numpy_1d(true_u)

    aligned, _, _, corr, rmse = align_recovered_u_to_true_u(mu_np, true_np)

    return rmse, corr, abs(corr) if np.isfinite(corr) else np.nan


def latent_distance_coherence(z, mu_u, max_pairs=20000):
    z_np = z.detach().cpu().numpy()
    u_np = mu_u.detach().cpu().numpy()

    n = z_np.shape[0]

    pairs = []

    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((i, j))

    if len(pairs) > max_pairs:
        chosen = np.random.choice(len(pairs), size=max_pairs, replace=False)
        pairs = [pairs[k] for k in chosen]

    dz = []
    du = []

    for i, j in pairs:
        dz.append(np.linalg.norm(z_np[i] - z_np[j]))
        du.append(np.linalg.norm(u_np[i] - u_np[j]))

    return safe_corr(dz, du)


def knn_latent_consistency(z, mu_u, k=5):
    z_np = z.detach().cpu().numpy()
    u_np = mu_u.detach().cpu().numpy()

    n = z_np.shape[0]

    if n <= k:
        return np.nan

    values = []

    for i in range(n):
        dz = np.linalg.norm(z_np - z_np[i], axis=1)
        idx = np.argsort(dz)[1:k + 1]

        du = np.linalg.norm(u_np[idx] - u_np[i], axis=1)
        values.append(np.mean(du))

    return float(np.mean(values))


# ============================================================
# 8. Training and evaluation
# ============================================================
def train_intervalgpvae_ablation_model(
    model,
    loader,
    device="cpu",
    joint_epochs=200,
    head_epochs=100,
    vae_refine_epochs=50,
    lr_joint=1e-3,
    lr_head=1e-3,
    lr_vae_refine=1e-4,
):
    # --------------------------------------------------------
    # Stage 1: joint training with AdamW
    # --------------------------------------------------------
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr_joint,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(joint_epochs):
        model.train()

        for batch in loader:
            z_batch, t_batch, y_batch, _ = [b.to(device) for b in batch]

            loss, info = model(z_batch, t_batch, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # --------------------------------------------------------
    # Stage 2: frozen VAE, train causal head with AdamW
    # --------------------------------------------------------
    for param in model.vae.parameters():
        param.requires_grad = False

    for param in model.causal_head.parameters():
        param.requires_grad = True

    causal_optimizer = torch.optim.AdamW(
        model.causal_head.parameters(),
        lr=lr_head,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(head_epochs):
        model.train()

        for z_batch, t_batch, y_batch, _ in loader:
            z_batch = z_batch.to(device)
            t_batch = t_batch.to(device)
            y_batch = y_batch.to(device)

            with torch.no_grad():
                mu_u = model.vae.get_latent_stats(
                    z_batch,
                    var_name="u",
                )[0]

                if model.use_aux:
                    mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
                        z_batch,
                        var_name="ua1",
                    )

                    ua1_batch = model.vae.reparameterize(
                        mu_ua1,
                        std_ua1,
                    )
                else:
                    ua1_batch = None

            z_y_batch = make_z_y(model, z_batch)

            y_pred = model.causal_head(
                mu_u.detach(),
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch.detach() if ua1_batch is not None else None,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
            )

            causal_optimizer.zero_grad()
            causal_loss.backward()
            causal_optimizer.step()

    # --------------------------------------------------------
    # Stage 3: frozen causal head, refine VAE with AdamW
    # --------------------------------------------------------
    for param in model.causal_head.parameters():
        param.requires_grad = False

    for param in model.vae.parameters():
        param.requires_grad = True

    vae_optimizer = torch.optim.AdamW(
        model.vae.parameters(),
        lr=lr_vae_refine,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(vae_refine_epochs):
        model.train()

        for batch in loader:
            z_batch, t_batch, y_batch, _ = [b.to(device) for b in batch]

            z_y_batch = make_z_y(model, z_batch)

            loss_vae, vae_info = model.vae(z_batch)

            ua1_batch = (
                model.vae.sample_latent(z_batch, var_name="ua1")
                if model.use_aux
                else None
            )

            y_pred = model.causal_head(
                vae_info["u"],
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
                reduction="sum",
            )

            total_loss = loss_vae + causal_loss

            vae_optimizer.zero_grad()
            total_loss.backward()
            vae_optimizer.step()

    # Unfreeze all parameters for safety.
    for param in model.parameters():
        param.requires_grad = True

    return model


def evaluate_ablation_model(
    model,
    z_train,
    z_test,
    true_u_test,
    true_ite_np,
    ci=0.90,
    n_mc=100,
    device="cpu",
):
    model.eval()

    z_train = z_train.to(device)
    z_test = z_test.to(device)
    true_u_test = true_u_test.to(device)

    intervalgp_results = ite_space_intervalgp_head(
        model=model,
        z_train=z_train,
        z_test=z_test,
        n_samples=n_mc,
        lengthscale=GP_LENGTHSCALE,
        variance=GP_VARIANCE,
        min_noise=GP_NOISE,
    )

    ite_mean = to_numpy_1d(intervalgp_results["ite_mean_test"])
    ite_lower = to_numpy_1d(intervalgp_results["ite_lower_test"])
    ite_upper = to_numpy_1d(intervalgp_results["ite_upper_test"])

    metrics = interval_metrics(
        ite_mean=ite_mean,
        ite_lower=ite_lower,
        ite_upper=ite_upper,
        true_ite=true_ite_np,
        nominal=ci,
    )

    metrics["NLL"] = gaussian_nll_from_interval(
        ite_mean=ite_mean,
        ite_lower=ite_lower,
        ite_upper=ite_upper,
        true_ite=true_ite_np,
        ci=ci,
    )

    metrics["interval_score"] = interval_score(
        ite_lower=ite_lower,
        ite_upper=ite_upper,
        true_ite=true_ite_np,
        alpha=1.0 - ci,
    )

    u_smooth = intervalgp_results["u_test_smooth"]
    u_smooth_std = intervalgp_results["u_test_smooth_std"]
    u_raw = intervalgp_results["u_test_raw"]

    latent_rmse, latent_corr, latent_abs_corr = latent_rmse_and_corr(
        mu_u=u_smooth,
        true_u=true_u_test,
    )

    raw_latent_rmse, raw_latent_corr, raw_latent_abs_corr = latent_rmse_and_corr(
        mu_u=u_raw,
        true_u=true_u_test,
    )

    metrics["latent_RMSE"] = latent_rmse
    metrics["latent_corr"] = latent_corr
    metrics["latent_abs_corr"] = latent_abs_corr

    metrics["raw_latent_RMSE"] = raw_latent_rmse
    metrics["raw_latent_corr"] = raw_latent_corr
    metrics["raw_latent_abs_corr"] = raw_latent_abs_corr

    metrics["avg_smooth_u_std"] = float(
        torch.mean(u_smooth_std.detach().cpu()).item()
    )

    metrics["latent_distance_coherence"] = latent_distance_coherence(
        z=z_test,
        mu_u=u_smooth,
    )

    metrics["knn_latent_consistency"] = knn_latent_consistency(
        z=z_test,
        mu_u=u_smooth,
        k=5,
    )

    return metrics


# ============================================================
# 9. Main ablation configurations
# ============================================================
ablation_configs = [
    {
        "name": "A0_VAE_standard",
        "prior_type": "standard",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A1_input_dependent_Gaussian",
        "prior_type": "input_gaussian",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A2_current_interval_GP_consistent_with_second_code",
        "prior_type": "interval_gp",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A3_joint_minibatch_GP_NLL",
        "prior_type": "joint_gp",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A3_joint_minibatch_GP_KL",
        "prior_type": "joint_gp_kl",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A4_sparse_GP_M32",
        "prior_type": "sparse_gp",
        "prior_weight": 1.0,
        "inducing_M": 32,
    },
    {
        "name": "A4_sparse_GP_M64",
        "prior_type": "sparse_gp",
        "prior_weight": 1.0,
        "inducing_M": 64,
    },
    {
        "name": "A4_sparse_GP_M128",
        "prior_type": "sparse_gp",
        "prior_weight": 1.0,
        "inducing_M": 128,
    },
]


def get_z_y_dim(chosen_version, z_dim):
    if chosen_version == "z_to_y":
        return z_dim

    if chosen_version == "split_z_to_t_and_y":
        return z_dim // 2

    return None


def get_use_aux(chosen_version):
    return chosen_version == "u_aux"


# ============================================================
# 10. Run experiments
# ============================================================
all_ablation_results = []

all_combinations = list(
    itertools.product(
        enumerate(proxy_func_sets, start=1),
        enumerate(treatment_func_list, start=1),
        enumerate(outcome_func_list, start=1),
    )
)

if MAX_ITERATIONS is not None:
    all_combinations = all_combinations[:MAX_ITERATIONS]

iteration_id = 0

# ============================================================
# Ablation A revised:
# Keep A0/A1/A2/A3/A3-KL/A4 comparison from the first code,
# but make IntervalGP-VAE implementation consistent with
# the second uploaded code.
#
# Key revisions:
#   1) Keep latent prior / regularizer ablation:
#        A0: Standard VAE prior
#        A1: Input-dependent Gaussian prior
#        A2: IntervalGP regularizer
#        A3: Joint mini-batch GP NLL
#        A3-KL: Joint mini-batch GP KL
#        A4: Sparse inducing GP
#   2) Use the same IntervalGP-VAE parameters as the second code:
#        GP_LENGTHSCALE = 7
#        GP_VARIANCE = 2.0
#        GP_NOISE = 1e-4
#        hidden_dim = 64
#        latent_dim = 1
#        causal_head_hidden_dim = 64
#   3) Use AdamW instead of Adam.
#   4) Use the same three-stage training framework:
#        Stage 1: joint VAE + causal head training
#        Stage 2: frozen VAE, train causal head
#        Stage 3: frozen causal head, refine VAE
#   5) Use the same inference framework:
#        latent GP posterior smoothing
#        ITE-space full GP posterior for lower/upper/mean ITE.
# ============================================================

import os
import time
import random
import itertools
import inspect
import textwrap
import warnings
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from scipy.stats import norm
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings("ignore")


# ============================================================
# 0. Global settings
# ============================================================
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

torch.set_default_tensor_type("torch.FloatTensor")
torch.set_default_dtype(torch.float32)

OUTPUT_DIR = "ablation_A_intervalgpvae_consistent_with_second_code"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cpu"

GLOBAL_SEED = 420
SEED = GLOBAL_SEED

np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

# Set to a small number, e.g. 2, for debugging.
# Set to None to run all 24 combinations.
MAX_ITERATIONS = None

N_TRAIN = 1000
N_TEST = 200

BATCH_SIZE = 128
JOINT_EPOCHS = 200
HEAD_EPOCHS = 100
VAE_REFINE_EPOCHS = 50

LR_JOINT = 1e-3
LR_HEAD = 1e-3
LR_VAE_REFINE = 1e-4
WEIGHT_DECAY = 1e-5

INTERVALGPVAE_LATENT_DIM = 1
INTERVALGPVAE_HIDDEN_DIM = 64
CAUSAL_HEAD_HIDDEN_DIM = 64

# Exact GP parameters from the second uploaded code version.
GP_LENGTHSCALE = 7
GP_VARIANCE = 2.0
GP_NOISE = 1e-4

ITE_CI = 0.90
Z_VALUE_90 = 1.6448536269514722

# Use the second code's ITE sampling size.
N_MC = 100

# If full GP is too slow, set MAX_GP_POINTS = 300.
# If you want full GP over all training samples, keep None.
MAX_GP_POINTS = None

CHOSEN_VERSION = "u_aux"
# Options:
#   "u_aux"               : use auxiliary latent ua1 in outcome head
#   "z_to_y"              : use all z as z_y
#   "split_z_to_t_and_y"  : use last half of z as z_y
#   "plain_u"             : only u and t in outcome head


def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)


set_seed(SEED)


# ============================================================
# 1. Utility functions
# ============================================================
def to_numpy_1d(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    return np.asarray(x).reshape(-1)


def safe_corr(x, y):
    x = to_numpy_1d(x)
    y = to_numpy_1d(y)

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 2:
        return np.nan

    if np.std(x[valid]) == 0 or np.std(y[valid]) == 0:
        return np.nan

    return float(np.corrcoef(x[valid], y[valid])[0, 1])


def pehe_np(est_ite, true_ite):
    est_ite = to_numpy_1d(est_ite)
    true_ite = to_numpy_1d(true_ite)

    valid = np.isfinite(est_ite) & np.isfinite(true_ite)

    if valid.sum() == 0:
        return np.nan

    return float(np.sqrt(np.mean((est_ite[valid] - true_ite[valid]) ** 2)))


def print_function_source(title, funcs):
    print(f"\n{title}:")

    if isinstance(funcs, list):
        for i, f in enumerate(funcs):
            print(f"  Function {i + 1}:")
            try:
                src = inspect.getsource(f).strip()
                print(textwrap.indent(src, "    "))
            except Exception:
                print("    [source not available]")
    else:
        try:
            src = inspect.getsource(funcs).strip()
            print(textwrap.indent(src, "  "))
        except Exception:
            print("  [source not available]")


def maybe_subsample_gp_reference(x, *ys, max_points=None, seed=0):
    n = x.shape[0]

    if max_points is None or n <= max_points:
        return (x, *ys)

    rng = np.random.default_rng(seed)
    idx = rng.choice(np.arange(n), size=max_points, replace=False)
    idx = np.sort(idx)

    idx_t = torch.tensor(idx, dtype=torch.long, device=x.device)

    out = [x[idx_t]]

    for y in ys:
        if torch.is_tensor(y):
            out.append(y[idx_t])
        else:
            out.append(np.asarray(y)[idx])

    return tuple(out)


# ============================================================
# 2. Synthetic data generation
# ============================================================
proxy_func_sets = [
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 2,
        lambda u, ua0, ua1: u ** 3,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.exp(-0.5 * u ** 2),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(0.5 * u),
        lambda u, ua0, ua1: torch.exp(0.3 * u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
        lambda u, ua0, ua1: torch.sigmoid(u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 2,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.exp(-0.2 * u ** 2),
        lambda u, ua0, ua1: torch.sin(2 * u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 3,
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
        lambda u, ua0, ua1: torch.sin(u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.tanh(0.5 * u),
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.exp(-0.3 * u ** 2),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: u ** 2,
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.log1p(torch.exp(u)),
        lambda u, ua0, ua1: torch.exp(0.5 * u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
    ],
]


def treat_func_linear_1(u_np: np.ndarray, ua0_np=None) -> np.ndarray:
    logits = 0.8 * u_np
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


def treat_func_linear_2(u_np: np.ndarray, ua0_np=None) -> np.ndarray:
    logits = 0.4 * u_np + 0.3
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


treatment_func_list = [
    treat_func_linear_1,
    treat_func_linear_2,
]


def outcome_func_1(
    u_np: np.ndarray,
    t_np: np.ndarray,
    ua0=None,
    ua1=None,
    add_noise=True,
) -> np.ndarray:
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    base = (
        0.5 * u_np
        + 0.2 * np.sin(u_np)
        + t_np.reshape(-1, 1) * (0.8 + 0.3 * u_np)
    )

    if add_noise:
        noise = np.random.normal(0.0, 0.1, size=base.shape)
        base = base + noise

    return base.astype(np.float32)


def outcome_func_2(
    u_np: np.ndarray,
    t_np: np.ndarray,
    ua0=None,
    ua1=None,
    add_noise=True,
) -> np.ndarray:
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    base = (
        0.5 * u_np
        + 0.3 * np.cos(u_np)
        + t_np.reshape(-1, 1) * (0.5 + 0.5 * u_np)
    )

    if add_noise:
        noise = np.random.normal(0.0, 0.1, size=base.shape)
        base = base + noise

    return base.astype(np.float32)


outcome_func_list = [
    outcome_func_1,
    outcome_func_2,
]


def call_outcome_func(
    outcome_func,
    u_np,
    t_np,
    z_np=None,
    ua0_np=None,
    ua1_np=None,
    add_noise=True,
):
    if isinstance(u_np, torch.Tensor):
        u_np = u_np.detach().cpu().numpy()
    if isinstance(t_np, torch.Tensor):
        t_np = t_np.detach().cpu().numpy()
    if ua0_np is not None and isinstance(ua0_np, torch.Tensor):
        ua0_np = ua0_np.detach().cpu().numpy()
    if ua1_np is not None and isinstance(ua1_np, torch.Tensor):
        ua1_np = ua1_np.detach().cpu().numpy()

    return outcome_func(
        u_np,
        t_np,
        ua0=ua0_np,
        ua1=ua1_np,
        add_noise=add_noise,
    )


def generate_synthetic_data_with_aux_uas(
    n=1000,
    noise_std=0.1,
    seed=0,
    num_proxies=None,
    proxy_funcs=None,
    outcome_func=None,
    treatment_func=None,
):
    set_seed(seed)

    u_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    ua0_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    ua1_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)

    u_tensor = torch.from_numpy(u_np)
    ua0_tensor = torch.from_numpy(ua0_np)
    ua1_tensor = torch.from_numpy(ua1_np)

    if proxy_funcs is None:
        proxy_funcs = proxy_func_sets[0]

    if num_proxies is None:
        num_proxies = len(proxy_funcs)

    proxy_funcs = proxy_funcs[:num_proxies]

    clean_z = torch.cat(
        [g(u_tensor, ua0_tensor, ua1_tensor) for g in proxy_funcs],
        dim=1,
    )

    eps_z = torch.randn(n, num_proxies) * noise_std
    z_tensor = clean_z + eps_z

    if treatment_func is None:
        treatment_func = treat_func_linear_1

    t_np = treatment_func(u_np, ua0_np)
    t_tensor = torch.from_numpy(t_np.astype(np.float32))

    if outcome_func is None:
        outcome_func = outcome_func_1

    y_np = outcome_func(
        u_np,
        t_np,
        ua0=ua0_np,
        ua1=ua1_np,
        add_noise=True,
    )

    y_tensor = torch.from_numpy(y_np.squeeze()).float()

    return {
        "z": z_tensor.float(),
        "t": t_tensor.float(),
        "y": y_tensor.float(),
        "u": u_tensor.squeeze().float(),
        "ua0": ua0_tensor.squeeze().float(),
        "ua1": ua1_tensor.squeeze().float(),
        "proxy_funcs": proxy_funcs,
        "outcome_func": outcome_func,
        "treatment_func": treatment_func,
    }


# ============================================================
# 3. Full GP posterior functions from second code framework
# ============================================================
def rbf_kernel(x1, x2=None, lengthscale=1.0, variance=1.0):
    if x2 is None:
        x2 = x1

    if x1.ndim == 1:
        x1 = x1.view(-1, 1)

    if x2.ndim == 1:
        x2 = x2.view(-1, 1)

    dists = torch.cdist(x1, x2).pow(2)

    return variance * torch.exp(-0.5 * dists / (lengthscale ** 2))


def full_gp_posterior_point(
    x_train,
    y_train,
    x_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
    jitter=1e-5,
):
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + noise * torch.eye(n_train, device=x_train.device)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(
                    n_train,
                    device=x_train.device,
                )
            )
            break
        except RuntimeError:
            current_jitter *= 10.0
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_point.")

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v

    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def full_gp_posterior_heteroscedastic(
    x_train,
    y_train,
    x_test,
    noise_var_train,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
    jitter=1e-5,
):
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    noise_var_train = noise_var_train.float().view(-1)
    noise_var_train = torch.clamp(noise_var_train, min=min_noise)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + torch.diag(noise_var_train)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(
                    n_train,
                    device=x_train.device,
                )
            )
            break
        except RuntimeError:
            current_jitter *= 10.0
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_heteroscedastic.")

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v

    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def latent_gp_posterior_smoothing(
    z_train,
    mu_u_train,
    z_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
):
    if mu_u_train.ndim == 1:
        mu_u_train = mu_u_train.view(-1, 1)

    latent_dim = mu_u_train.shape[1]
    means = []
    vars_diag = []

    for r in range(latent_dim):
        mean_r, cov_r = full_gp_posterior_point(
            x_train=z_train,
            y_train=mu_u_train[:, r],
            x_test=z_test,
            lengthscale=lengthscale,
            variance=variance,
            noise=noise,
        )

        var_r = torch.diag(cov_r).clamp_min(1e-8)

        means.append(mean_r)
        vars_diag.append(var_r)

    mean = torch.stack(means, dim=1)
    std = torch.sqrt(torch.stack(vars_diag, dim=1))

    return mean, std


# ============================================================
# 4. Model components
# ============================================================
class DoubleEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.mean_u = nn.Linear(hidden_dim, latent_dim)
        self.logvar_u = nn.Linear(hidden_dim, latent_dim)

        self.mean_eps = nn.Linear(hidden_dim, input_dim)
        self.logvar_eps = nn.Linear(hidden_dim, input_dim)

        if self.use_aux:
            self.mean_ua0 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua0 = nn.Linear(hidden_dim, latent_dim)

            self.mean_ua1 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua1 = nn.Linear(hidden_dim, latent_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))

        mu_u = self.mean_u(h)
        logvar_u = self.logvar_u(h)
        std_u = torch.exp(0.5 * logvar_u)

        mu_eps = self.mean_eps(h)
        logvar_eps = self.logvar_eps(h)
        std_eps = torch.exp(0.5 * logvar_eps)

        if self.use_aux:
            mu_ua0 = self.mean_ua0(h)
            logvar_ua0 = self.logvar_ua0(h)
            std_ua0 = torch.exp(0.5 * logvar_ua0)

            mu_ua1 = self.mean_ua1(h)
            logvar_ua1 = self.logvar_ua1(h)
            std_ua1 = torch.exp(0.5 * logvar_ua1)

            return (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            )

        return mu_u, std_u, logvar_u, mu_eps, std_eps, logvar_eps


class AdditiveDecoder(nn.Module):
    def __init__(
        self,
        latent_dim,
        hidden_dim,
        output_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        factor = 1 + (2 if use_auxiliary_latents else 0)

        self.fc1 = nn.Linear(latent_dim * factor, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, u, eps, ua0=None, ua1=None):
        if self.use_aux:
            if ua0 is None or ua1 is None:
                raise ValueError("ua0 and ua1 are required when use_auxiliary_latents=True.")
            x = torch.cat([u, ua0, ua1], dim=1)
        else:
            x = u

        h = F.relu(self.fc1(x))

        return self.out(h) + eps


class FlexibleCausalHead(nn.Module):
    def __init__(
        self,
        latent_dim_u,
        z_y_dim=None,
        use_auxiliary_latents=False,
        hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        self.z_y_dim = z_y_dim

        input_dim = latent_dim_u + 1

        if z_y_dim is not None:
            input_dim += z_y_dim

        if use_auxiliary_latents:
            input_dim += latent_dim_u

        self.fc = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, u, t, z_y=None, ua1=None):
        t = t.view(-1, 1)

        parts = [u, t]

        if self.z_y_dim is not None:
            if z_y is None:
                raise ValueError("z_y is required but got None.")
            parts.append(z_y)

        if self.use_aux:
            if ua1 is None:
                raise ValueError("ua1 is required but got None.")
            parts.append(ua1)

        x = torch.cat(parts, dim=1)
        h = F.relu(self.fc(x))

        return self.out(h)


# ============================================================
# 5. Ablation-ready VAE backbone, but with second-code framework
# ============================================================
class GPVAEwithNoise(nn.Module):
    """
    prior_type:
        "standard"        : A0, standard VAE prior N(0,I)
        "input_gaussian"  : A1, input-dependent Gaussian prior
        "interval_gp"     : A2, IntervalGP regularizer
        "joint_gp"        : A3, joint mini-batch GP NLL
        "joint_gp_kl"     : A3-KL, joint mini-batch GP KL
        "sparse_gp"       : A4, sparse inducing GP approximation
    """

    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        use_auxiliary_latents=False,
        prior_type="interval_gp",
        prior_weight=1.0,
        std_penalty_weight=10.0,
        inducing_points=None,
        input_prior_net_hidden=32,
        jitter=1e-5,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.encoder = DoubleEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.decoder = AdditiveDecoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            output_dim=input_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.gp_lengthscale = gp_lengthscale
        self.gp_variance = gp_variance
        self.gp_noise = gp_noise

        self.prior_type = prior_type
        self.prior_weight = prior_weight
        self.std_penalty_weight = std_penalty_weight
        self.jitter = jitter

        self.input_prior_net = nn.Sequential(
            nn.Linear(input_dim, input_prior_net_hidden),
            nn.ReLU(),
            nn.Linear(input_prior_net_hidden, latent_dim),
        )

        if inducing_points is not None:
            self.register_buffer("inducing_points", inducing_points.clone().float())
        else:
            self.inducing_points = None

    def reparameterize(self, mu, std):
        return mu + torch.randn_like(std) * std

    def rbf_kernel(self, x1, x2=None):
        if x2 is None:
            x2 = x1

        if x1.ndim == 1:
            x1 = x1.view(-1, 1)

        if x2.ndim == 1:
            x2 = x2.view(-1, 1)

        dists = torch.cdist(x1, x2).pow(2)

        return self.gp_variance * torch.exp(
            -0.5 * dists / (self.gp_lengthscale ** 2)
        )

    def stable_cholesky(self, K, max_tries=8):
        eye = torch.eye(K.shape[0], device=K.device, dtype=K.dtype)
        jitter = self.jitter

        for _ in range(max_tries):
            try:
                return torch.linalg.cholesky(K + jitter * eye)
            except RuntimeError:
                jitter *= 10.0

        raise RuntimeError("Cholesky decomposition failed.")

    # ------------------------------
    # A0
    # ------------------------------
    def standard_normal_kl(self, mu_u, logvar_u):
        return -0.5 * torch.sum(
            1.0 + logvar_u - mu_u.pow(2) - logvar_u.exp()
        )

    # ------------------------------
    # A1
    # ------------------------------
    def input_dependent_gaussian_kl(self, z, mu_u, std_u, logvar_u):
        log_s2 = self.input_prior_net(z)
        log_s2 = torch.clamp(log_s2, min=-8.0, max=8.0)
        s2 = torch.exp(log_s2)

        q_var = std_u.pow(2)

        kl = 0.5 * torch.sum(
            log_s2 - logvar_u + (q_var + mu_u.pow(2)) / s2 - 1.0
        )

        return kl

    # ------------------------------
    # A2
    # ------------------------------
    def interval_gp_regularizer(self, z, mu_u, std_u):
        """
        This keeps the A2 per-sample IntervalGP regularizer,
        but uses the GP parameters from the second code.
        Full GP posterior is applied at inference.
        """
        marginal_var = self.gp_variance + self.gp_noise
        marginal_std = torch.sqrt(
            torch.tensor(marginal_var, device=z.device, dtype=z.dtype)
        )

        lower = mu_u - std_u
        upper = mu_u + std_u

        dist = torch.distributions.Normal(
            loc=torch.zeros_like(mu_u),
            scale=marginal_std * torch.ones_like(mu_u),
        )

        prob = dist.cdf(upper) - dist.cdf(lower)
        prob = torch.clamp(prob, min=1e-8)

        return -torch.sum(torch.log(prob))

    # ------------------------------
    # A3
    # ------------------------------
    def joint_gp_nll(self, z, mu_u):
        B, d = mu_u.shape

        K = self.rbf_kernel(z, z)
        K = K + (self.gp_noise + self.jitter) * torch.eye(
            B,
            device=z.device,
            dtype=z.dtype,
        )

        L = self.stable_cholesky(K)

        loss = 0.0

        for j in range(d):
            y = mu_u[:, j:j + 1]
            alpha = torch.cholesky_solve(y, L)

            quad = 0.5 * (y.T @ alpha).squeeze()
            logdet = torch.sum(torch.log(torch.diag(L)))
            const = 0.5 * B * np.log(2.0 * np.pi)

            loss = loss + quad + logdet + const

        return loss

    # ------------------------------
    # A3-KL
    # ------------------------------
    def joint_gp_kl(self, z, mu_u, std_u):
        B, d = mu_u.shape

        K = self.rbf_kernel(z, z)
        K = K + (self.gp_noise + self.jitter) * torch.eye(
            B,
            device=z.device,
            dtype=z.dtype,
        )

        L = self.stable_cholesky(K)

        K_inv = torch.cholesky_inverse(L)
        K_inv_diag = K_inv.diag()
        logdet_K = 2.0 * torch.sum(torch.log(torch.diag(L)))

        loss = 0.0

        for j in range(d):
            mu_j = mu_u[:, j:j + 1]
            var_j = std_u[:, j].pow(2) + 1e-8

            alpha = torch.cholesky_solve(mu_j, L)
            quad = (mu_j.T @ alpha).squeeze()
            trace = torch.sum(K_inv_diag * var_j)
            logdet_q = torch.sum(torch.log(var_j))

            kl_j = 0.5 * (trace + quad - B + logdet_K - logdet_q)
            loss = loss + kl_j

        return loss

    # ------------------------------
    # A4
    # ------------------------------
    def sparse_gp_nll(self, z, mu_u):
        if self.inducing_points is None:
            raise ValueError("inducing_points must be provided for sparse_gp.")

        B, d = mu_u.shape

        Zm = self.inducing_points.to(device=z.device, dtype=z.dtype)

        K_mm = self.rbf_kernel(Zm, Zm)
        K_mm = K_mm + self.jitter * torch.eye(
            K_mm.shape[0],
            device=z.device,
            dtype=z.dtype,
        )

        L_mm = self.stable_cholesky(K_mm)

        K_bm = self.rbf_kernel(z, Zm)
        K_mb = K_bm.T

        Kmm_inv_Kmb = torch.cholesky_solve(K_mb, L_mm)
        Q_bb = K_bm @ Kmm_inv_Kmb

        K_approx = Q_bb + (self.gp_noise + self.jitter) * torch.eye(
            B,
            device=z.device,
            dtype=z.dtype,
        )

        L = self.stable_cholesky(K_approx)

        loss = 0.0

        for j in range(d):
            y = mu_u[:, j:j + 1]
            alpha = torch.cholesky_solve(y, L)

            quad = 0.5 * (y.T @ alpha).squeeze()
            logdet = torch.sum(torch.log(torch.diag(L)))
            const = 0.5 * B * np.log(2.0 * np.pi)

            loss = loss + quad + logdet + const

        return loss

    def latent_prior_regularizer(self, z, mu_u, std_u, logvar_u):
        if self.prior_type == "standard":
            return self.standard_normal_kl(mu_u, logvar_u)

        if self.prior_type == "input_gaussian":
            return self.input_dependent_gaussian_kl(z, mu_u, std_u, logvar_u)

        if self.prior_type == "interval_gp":
            return self.interval_gp_regularizer(z, mu_u, std_u)

        if self.prior_type == "joint_gp":
            return self.joint_gp_nll(z, mu_u)

        if self.prior_type == "joint_gp_kl":
            return self.joint_gp_kl(z, mu_u, std_u)

        if self.prior_type == "sparse_gp":
            return self.sparse_gp_nll(z, mu_u)

        raise ValueError(f"Unknown prior_type: {self.prior_type}")

    def get_latent_stats(self, z, var_name="u"):
        outputs = self.encoder(z)

        if var_name == "u":
            return outputs[0], outputs[1], outputs[2]

        if var_name == "eps":
            return outputs[3], outputs[4], outputs[5]

        if var_name == "ua0":
            if not self.use_aux:
                return None, None, None
            return outputs[6], outputs[7], outputs[8]

        if var_name == "ua1":
            if not self.use_aux:
                return None, None, None
            return outputs[9], outputs[10], outputs[11]

        raise ValueError(f"Invalid var_name: {var_name}")

    def sample_latent(self, z, var_name="u"):
        mu, std, _ = self.get_latent_stats(z, var_name=var_name)

        if mu is None:
            return None

        return self.reparameterize(mu, std)

    def forward(self, z):
        if self.use_aux:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
                mu_ua0,
                std_ua0,
                logvar_ua0,
                mu_ua1,
                std_ua1,
                logvar_ua1,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = self.reparameterize(mu_ua0, std_ua0)
            ua1 = self.reparameterize(mu_ua1, std_ua1)

            z_recon = self.decoder(u, eps, ua0=ua0, ua1=ua1)

        else:
            (
                mu_u,
                std_u,
                logvar_u,
                mu_eps,
                std_eps,
                logvar_eps,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = None
            ua1 = None

            z_recon = self.decoder(u, eps)

        recon_loss = F.mse_loss(z_recon, z, reduction="sum")

        prior_loss = self.latent_prior_regularizer(
            z=z,
            mu_u=mu_u,
            std_u=std_u,
            logvar_u=logvar_u,
        )

        kl_eps = -0.5 * torch.sum(
            1.0 + logvar_eps - mu_eps.pow(2) - logvar_eps.exp()
        )

        std_penalty = torch.sum(std_u ** 2)

        total_loss = (
            recon_loss
            + self.prior_weight * prior_loss
            + kl_eps
            + self.std_penalty_weight * std_penalty
        )

        if self.use_aux:
            kl_ua0 = -0.5 * torch.sum(
                1.0 + logvar_ua0 - mu_ua0.pow(2) - logvar_ua0.exp()
            )

            kl_ua1 = -0.5 * torch.sum(
                1.0 + logvar_ua1 - mu_ua1.pow(2) - logvar_ua1.exp()
            )

            total_loss = total_loss + kl_ua0 + kl_ua1

        return total_loss, {
            "recon_loss": float(recon_loss.detach().cpu()),
            "prior_loss": float(prior_loss.detach().cpu()),
            "kl_eps": float(kl_eps.detach().cpu()),
            "std_penalty": float(std_penalty.detach().cpu()),
            "u": u,
            "eps": eps,
            "ua0": ua0,
            "ua1": ua1,
            "mu_u": mu_u,
            "std_u": std_u,
            "z_recon": z_recon,
        }


class CausalGPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        z_y_dim=None,
        use_auxiliary_latents=False,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        prior_type="interval_gp",
        prior_weight=1.0,
        std_penalty_weight=10.0,
        inducing_points=None,
    ):
        super().__init__()

        self.z_y_dim = z_y_dim
        self.use_aux = use_auxiliary_latents

        self.vae = GPVAEwithNoise(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            gp_lengthscale=gp_lengthscale,
            gp_variance=gp_variance,
            gp_noise=gp_noise,
            use_auxiliary_latents=use_auxiliary_latents,
            prior_type=prior_type,
            prior_weight=prior_weight,
            std_penalty_weight=std_penalty_weight,
            inducing_points=inducing_points,
        )

        self.causal_head = FlexibleCausalHead(
            latent_dim_u=latent_dim,
            z_y_dim=z_y_dim,
            use_auxiliary_latents=use_auxiliary_latents,
            hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
        )

    def forward(self, z, t, y):
        loss_vae, vae_info = self.vae(z)

        u = vae_info["u"]
        ua1 = vae_info["ua1"] if self.use_aux else None

        z_y = make_z_y(self, z)

        y_pred = self.causal_head(
            u,
            t,
            z_y=z_y,
            ua1=ua1,
        )

        causal_loss = F.mse_loss(
            y_pred,
            y.unsqueeze(-1),
            reduction="sum",
        )

        total_loss = loss_vae + causal_loss

        return total_loss, {
            **vae_info,
            "y_pred": y_pred,
            "causal_loss": float(causal_loss.detach().cpu()),
        }


def make_z_y(model, z):
    if model.causal_head.z_y_dim == z.shape[1]:
        return z

    if model.causal_head.z_y_dim:
        return z[:, z.shape[1] // 2:]

    return None


# ============================================================
# 6. ITE-space IntervalGP inference from second code
# ============================================================
def sample_ite_from_encoder(
    model,
    z,
    n_samples=100,
):
    model.eval()

    with torch.no_grad():
        mu_u, std_u, _ = model.vae.get_latent_stats(z, var_name="u")
        N, d_u = mu_u.shape
        device = z.device

        z_y = make_z_y(model, z)

        mu_ua1, std_ua1, _ = model.vae.get_latent_stats(z, var_name="ua1")
        has_ua1 = (mu_ua1 is not None) and (std_ua1 is not None)

        eps_u = torch.randn(n_samples, N, d_u, device=device)
        u_samples = mu_u.unsqueeze(0) + eps_u * std_u.unsqueeze(0)

        U = u_samples.reshape(-1, d_u)

        if has_ua1:
            d_ua = mu_ua1.shape[1]
            eps_ua1 = torch.randn(n_samples, N, d_ua, device=device)
            ua1_samples = mu_ua1.unsqueeze(0) + eps_ua1 * std_ua1.unsqueeze(0)
            UA1 = ua1_samples.reshape(-1, d_ua)
        else:
            UA1 = None

        if z_y is not None:
            ZY = (
                z_y.unsqueeze(0)
                .expand(n_samples, -1, -1)
                .reshape(-1, z_y.shape[1])
            )
        else:
            ZY = None

        t0 = torch.zeros(U.shape[0], device=device)
        t1 = torch.ones(U.shape[0], device=device)

        y0 = model.causal_head(U, t0, z_y=ZY, ua1=UA1).squeeze(-1)
        y1 = model.causal_head(U, t1, z_y=ZY, ua1=UA1).squeeze(-1)

        ite_samples = (y1 - y0).view(n_samples, N)

        lower_q = (1.0 - ITE_CI) / 2.0
        upper_q = 1.0 - lower_q

        ite_mean = ite_samples.mean(dim=0)
        ite_std = ite_samples.std(dim=0)
        ite_lower = ite_samples.quantile(lower_q, dim=0)
        ite_upper = ite_samples.quantile(upper_q, dim=0)

        return ite_samples, ite_mean, ite_std, ite_lower, ite_upper


def ite_space_intervalgp_head(
    model,
    z_train,
    z_test,
    n_samples=100,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
):
    model.eval()

    with torch.no_grad():
        _, ite_mean_train, ite_std_train, ite_lower_train, ite_upper_train = (
            sample_ite_from_encoder(
                model=model,
                z=z_train,
                n_samples=n_samples,
            )
        )

        mu_u_train, _, _ = model.vae.get_latent_stats(z_train, var_name="u")
        mu_u_test, _, _ = model.vae.get_latent_stats(z_test, var_name="u")

        # Latent GP posterior smoothing before ITE-space GP.
        u_test_smooth, u_test_smooth_std = latent_gp_posterior_smoothing(
            z_train=z_train,
            mu_u_train=mu_u_train,
            z_test=z_test,
            lengthscale=lengthscale,
            variance=variance,
            noise=min_noise,
        )

        u_train_smooth = mu_u_train.detach()

        (
            u_train_gp,
            ite_lower_train_gp,
            ite_upper_train_gp,
            ite_mean_train_gp,
            ite_std_train_gp,
        ) = maybe_subsample_gp_reference(
            u_train_smooth,
            ite_lower_train.detach(),
            ite_upper_train.detach(),
            ite_mean_train.detach(),
            ite_std_train.detach(),
            max_points=MAX_GP_POINTS,
            seed=123,
        )

        noise_var_lower = ite_std_train_gp.pow(2) + min_noise
        noise_var_upper = ite_std_train_gp.pow(2) + min_noise
        noise_var_mean = ite_std_train_gp.pow(2) + min_noise

        gp_lower_mean, gp_lower_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_lower_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_lower,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_upper_mean, gp_upper_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_upper_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_upper,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_ite_mean, gp_ite_cov = full_gp_posterior_heteroscedastic(
            x_train=u_train_gp,
            y_train=ite_mean_train_gp,
            x_test=u_test_smooth,
            noise_var_train=noise_var_mean,
            lengthscale=lengthscale,
            variance=variance,
            min_noise=min_noise,
        )

        gp_lower_std = torch.sqrt(torch.diag(gp_lower_cov).clamp_min(1e-8))
        gp_upper_std = torch.sqrt(torch.diag(gp_upper_cov).clamp_min(1e-8))
        gp_ite_std = torch.sqrt(torch.diag(gp_ite_cov).clamp_min(1e-8))

        raw_lower = torch.minimum(gp_lower_mean, gp_upper_mean)
        raw_upper = torch.maximum(gp_lower_mean, gp_upper_mean)

        ite_lower_test = raw_lower - Z_VALUE_90 * gp_lower_std
        ite_upper_test = raw_upper + Z_VALUE_90 * gp_upper_std

        return {
            "ite_mean_test": gp_ite_mean,
            "ite_std_test": gp_ite_std,
            "ite_lower_test": ite_lower_test,
            "ite_upper_test": ite_upper_test,
            "u_test_smooth": u_test_smooth,
            "u_test_smooth_std": u_test_smooth_std,
            "u_test_raw": mu_u_test,
            "ite_lower_train": ite_lower_train,
            "ite_upper_train": ite_upper_train,
            "ite_mean_train": ite_mean_train,
            "ite_std_train": ite_std_train,
        }


# ============================================================
# 7. Metrics
# ============================================================
def interval_metrics(ite_mean, ite_lower, ite_upper, true_ite, nominal=0.90):
    ite_mean = np.asarray(ite_mean).reshape(-1)
    ite_lower = np.asarray(ite_lower).reshape(-1)
    ite_upper = np.asarray(ite_upper).reshape(-1)
    true_ite = np.asarray(true_ite).reshape(-1)

    pehe = float(np.sqrt(np.mean((ite_mean - true_ite) ** 2)))

    ate_est = float(np.mean(ite_mean))
    ate_true = float(np.mean(true_ite))
    ate_error = float(np.abs(ate_est - ate_true))

    covered = (ite_lower <= true_ite) & (true_ite <= ite_upper)
    coverage = float(np.mean(covered))

    width = float(np.mean(ite_upper - ite_lower))
    calibration_error = float(np.abs(coverage - nominal))

    return {
        "PEHE": pehe,
        "ATE_est": ate_est,
        "ATE_true": ate_true,
        "ATE_error": ate_error,
        "coverage": coverage,
        "interval_width": width,
        "calibration_error": calibration_error,
    }


def gaussian_nll_from_interval(ite_mean, ite_lower, ite_upper, true_ite, ci=0.90):
    z_value = norm.ppf(0.5 + ci / 2.0)

    ite_mean = np.asarray(ite_mean).reshape(-1)
    ite_lower = np.asarray(ite_lower).reshape(-1)
    ite_upper = np.asarray(ite_upper).reshape(-1)
    true_ite = np.asarray(true_ite).reshape(-1)

    std = (ite_upper - ite_lower) / (2.0 * z_value)
    std = np.maximum(std, 1e-6)

    nll = (
        0.5 * np.log(2.0 * np.pi * std ** 2)
        + 0.5 * ((true_ite - ite_mean) / std) ** 2
    )

    return float(np.mean(nll))


def interval_score(ite_lower, ite_upper, true_ite, alpha=0.10):
    lower = np.asarray(ite_lower).reshape(-1)
    upper = np.asarray(ite_upper).reshape(-1)
    y = np.asarray(true_ite).reshape(-1)

    width = upper - lower
    below = y < lower
    above = y > upper

    score = (
        width
        + (2.0 / alpha) * (lower - y) * below
        + (2.0 / alpha) * (y - upper) * above
    )

    return float(np.mean(score))


def align_recovered_u_to_true_u(u_recovered, u_true):
    u_recovered = to_numpy_1d(u_recovered)
    u_true = to_numpy_1d(u_true)

    valid = np.isfinite(u_recovered) & np.isfinite(u_true)

    if valid.sum() < 2:
        return u_recovered, np.nan, np.nan, np.nan, np.nan

    x = u_recovered[valid]
    y = u_true[valid]

    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]

    u_aligned = a * u_recovered + b

    corr = safe_corr(u_aligned, u_true)
    rmse = float(np.sqrt(np.mean((u_aligned[valid] - u_true[valid]) ** 2)))

    return u_aligned, float(a), float(b), corr, rmse


def latent_rmse_and_corr(mu_u, true_u):
    mu_np = to_numpy_1d(mu_u)
    true_np = to_numpy_1d(true_u)

    aligned, _, _, corr, rmse = align_recovered_u_to_true_u(mu_np, true_np)

    return rmse, corr, abs(corr) if np.isfinite(corr) else np.nan


def latent_distance_coherence(z, mu_u, max_pairs=20000):
    z_np = z.detach().cpu().numpy()
    u_np = mu_u.detach().cpu().numpy()

    n = z_np.shape[0]

    pairs = []

    for i in range(n):
        for j in range(i + 1, n):
            pairs.append((i, j))

    if len(pairs) > max_pairs:
        chosen = np.random.choice(len(pairs), size=max_pairs, replace=False)
        pairs = [pairs[k] for k in chosen]

    dz = []
    du = []

    for i, j in pairs:
        dz.append(np.linalg.norm(z_np[i] - z_np[j]))
        du.append(np.linalg.norm(u_np[i] - u_np[j]))

    return safe_corr(dz, du)


def knn_latent_consistency(z, mu_u, k=5):
    z_np = z.detach().cpu().numpy()
    u_np = mu_u.detach().cpu().numpy()

    n = z_np.shape[0]

    if n <= k:
        return np.nan

    values = []

    for i in range(n):
        dz = np.linalg.norm(z_np - z_np[i], axis=1)
        idx = np.argsort(dz)[1:k + 1]

        du = np.linalg.norm(u_np[idx] - u_np[i], axis=1)
        values.append(np.mean(du))

    return float(np.mean(values))


# ============================================================
# 8. Training and evaluation
# ============================================================
def train_intervalgpvae_ablation_model(
    model,
    loader,
    device="cpu",
    joint_epochs=200,
    head_epochs=100,
    vae_refine_epochs=50,
    lr_joint=1e-3,
    lr_head=1e-3,
    lr_vae_refine=1e-4,
):
    # --------------------------------------------------------
    # Stage 1: joint training with AdamW
    # --------------------------------------------------------
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr_joint,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(joint_epochs):
        model.train()

        for batch in loader:
            z_batch, t_batch, y_batch, _ = [b.to(device) for b in batch]

            loss, info = model(z_batch, t_batch, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # --------------------------------------------------------
    # Stage 2: frozen VAE, train causal head with AdamW
    # --------------------------------------------------------
    for param in model.vae.parameters():
        param.requires_grad = False

    for param in model.causal_head.parameters():
        param.requires_grad = True

    causal_optimizer = torch.optim.AdamW(
        model.causal_head.parameters(),
        lr=lr_head,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(head_epochs):
        model.train()

        for z_batch, t_batch, y_batch, _ in loader:
            z_batch = z_batch.to(device)
            t_batch = t_batch.to(device)
            y_batch = y_batch.to(device)

            with torch.no_grad():
                mu_u = model.vae.get_latent_stats(
                    z_batch,
                    var_name="u",
                )[0]

                if model.use_aux:
                    mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
                        z_batch,
                        var_name="ua1",
                    )

                    ua1_batch = model.vae.reparameterize(
                        mu_ua1,
                        std_ua1,
                    )
                else:
                    ua1_batch = None

            z_y_batch = make_z_y(model, z_batch)

            y_pred = model.causal_head(
                mu_u.detach(),
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch.detach() if ua1_batch is not None else None,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
            )

            causal_optimizer.zero_grad()
            causal_loss.backward()
            causal_optimizer.step()

    # --------------------------------------------------------
    # Stage 3: frozen causal head, refine VAE with AdamW
    # --------------------------------------------------------
    for param in model.causal_head.parameters():
        param.requires_grad = False

    for param in model.vae.parameters():
        param.requires_grad = True

    vae_optimizer = torch.optim.AdamW(
        model.vae.parameters(),
        lr=lr_vae_refine,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(vae_refine_epochs):
        model.train()

        for batch in loader:
            z_batch, t_batch, y_batch, _ = [b.to(device) for b in batch]

            z_y_batch = make_z_y(model, z_batch)

            loss_vae, vae_info = model.vae(z_batch)

            ua1_batch = (
                model.vae.sample_latent(z_batch, var_name="ua1")
                if model.use_aux
                else None
            )

            y_pred = model.causal_head(
                vae_info["u"],
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
                reduction="sum",
            )

            total_loss = loss_vae + causal_loss

            vae_optimizer.zero_grad()
            total_loss.backward()
            vae_optimizer.step()

    # Unfreeze all parameters for safety.
    for param in model.parameters():
        param.requires_grad = True

    return model


def evaluate_ablation_model(
    model,
    z_train,
    z_test,
    true_u_test,
    true_ite_np,
    ci=0.90,
    n_mc=100,
    device="cpu",
):
    model.eval()

    z_train = z_train.to(device)
    z_test = z_test.to(device)
    true_u_test = true_u_test.to(device)

    intervalgp_results = ite_space_intervalgp_head(
        model=model,
        z_train=z_train,
        z_test=z_test,
        n_samples=n_mc,
        lengthscale=GP_LENGTHSCALE,
        variance=GP_VARIANCE,
        min_noise=GP_NOISE,
    )

    ite_mean = to_numpy_1d(intervalgp_results["ite_mean_test"])
    ite_lower = to_numpy_1d(intervalgp_results["ite_lower_test"])
    ite_upper = to_numpy_1d(intervalgp_results["ite_upper_test"])

    metrics = interval_metrics(
        ite_mean=ite_mean,
        ite_lower=ite_lower,
        ite_upper=ite_upper,
        true_ite=true_ite_np,
        nominal=ci,
    )

    metrics["NLL"] = gaussian_nll_from_interval(
        ite_mean=ite_mean,
        ite_lower=ite_lower,
        ite_upper=ite_upper,
        true_ite=true_ite_np,
        ci=ci,
    )

    metrics["interval_score"] = interval_score(
        ite_lower=ite_lower,
        ite_upper=ite_upper,
        true_ite=true_ite_np,
        alpha=1.0 - ci,
    )

    u_smooth = intervalgp_results["u_test_smooth"]
    u_smooth_std = intervalgp_results["u_test_smooth_std"]
    u_raw = intervalgp_results["u_test_raw"]

    latent_rmse, latent_corr, latent_abs_corr = latent_rmse_and_corr(
        mu_u=u_smooth,
        true_u=true_u_test,
    )

    raw_latent_rmse, raw_latent_corr, raw_latent_abs_corr = latent_rmse_and_corr(
        mu_u=u_raw,
        true_u=true_u_test,
    )

    metrics["latent_RMSE"] = latent_rmse
    metrics["latent_corr"] = latent_corr
    metrics["latent_abs_corr"] = latent_abs_corr

    metrics["raw_latent_RMSE"] = raw_latent_rmse
    metrics["raw_latent_corr"] = raw_latent_corr
    metrics["raw_latent_abs_corr"] = raw_latent_abs_corr

    metrics["avg_smooth_u_std"] = float(
        torch.mean(u_smooth_std.detach().cpu()).item()
    )

    metrics["latent_distance_coherence"] = latent_distance_coherence(
        z=z_test,
        mu_u=u_smooth,
    )

    metrics["knn_latent_consistency"] = knn_latent_consistency(
        z=z_test,
        mu_u=u_smooth,
        k=5,
    )

    return metrics


# ============================================================
# 9. Main ablation configurations
# ============================================================
ablation_configs = [
    {
        "name": "A0_VAE_standard",
        "prior_type": "standard",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A1_input_dependent_Gaussian",
        "prior_type": "input_gaussian",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A2_current_interval_GP_consistent_with_second_code",
        "prior_type": "interval_gp",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A3_joint_minibatch_GP_NLL",
        "prior_type": "joint_gp",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A3_joint_minibatch_GP_KL",
        "prior_type": "joint_gp_kl",
        "prior_weight": 1.0,
        "inducing_M": None,
    },
    {
        "name": "A4_sparse_GP_M32",
        "prior_type": "sparse_gp",
        "prior_weight": 1.0,
        "inducing_M": 32,
    },
    {
        "name": "A4_sparse_GP_M64",
        "prior_type": "sparse_gp",
        "prior_weight": 1.0,
        "inducing_M": 64,
    },
    {
        "name": "A4_sparse_GP_M128",
        "prior_type": "sparse_gp",
        "prior_weight": 1.0,
        "inducing_M": 128,
    },
]


def get_z_y_dim(chosen_version, z_dim):
    if chosen_version == "z_to_y":
        return z_dim

    if chosen_version == "split_z_to_t_and_y":
        return z_dim // 2

    return None


def get_use_aux(chosen_version):
    return chosen_version == "u_aux"


# ============================================================
# 10. Run experiments
# ============================================================
all_ablation_results = []

all_combinations = list(
    itertools.product(
        enumerate(proxy_func_sets, start=1),
        enumerate(treatment_func_list, start=1),
        enumerate(outcome_func_list, start=1),
    )
)

if MAX_ITERATIONS is not None:
    all_combinations = all_combinations[:MAX_ITERATIONS]

iteration_id = 0

# ============================================================
# 10. Run experiments continued
# ============================================================
for (proxy_id, proxy_funcs), (treat_id, treat_func), (outcome_id, outcome_func) in all_combinations:

    iteration_id += 1

    print("\n" + "=" * 100)
    print(f"Combination {iteration_id}")
    print(f"Proxy ID = {proxy_id}, Treatment ID = {treat_id}, Outcome ID = {outcome_id}")
    print("=" * 100)

    data_train = generate_synthetic_data_with_aux_uas(
        n=N_TRAIN,
        num_proxies=len(proxy_funcs),
        proxy_funcs=proxy_funcs,
        treatment_func=treat_func,
        outcome_func=outcome_func,
        seed=42,
    )

    data_test = generate_synthetic_data_with_aux_uas(
        n=N_TEST,
        num_proxies=len(proxy_funcs),
        proxy_funcs=proxy_funcs,
        treatment_func=treat_func,
        outcome_func=outcome_func,
        seed=100 + iteration_id,
    )

    z = data_train["z"].to(DEVICE)
    t = data_train["t"].to(DEVICE)
    y = data_train["y"].to(DEVICE)
    u_true = data_train["u"].to(DEVICE)

    z_test = data_test["z"].to(DEVICE)
    true_u_test = data_test["u"].to(DEVICE)

    true_ua0_test = data_test.get("ua0", None)
    true_ua1_test = data_test.get("ua1", None)

    if true_ua0_test is not None:
        ua0_test_np = true_ua0_test.cpu().numpy().reshape(-1, 1)
    else:
        ua0_test_np = None

    if true_ua1_test is not None:
        ua1_test_np = true_ua1_test.cpu().numpy().reshape(-1, 1)
    else:
        ua1_test_np = None

    # --------------------------------------------------------
    # Standardize Z using train statistics.
    # This follows the second-code framework more closely.
    # --------------------------------------------------------
    z_mean = z.mean(dim=0, keepdim=True)
    z_std = z.std(dim=0, keepdim=True).clamp_min(1e-6)

    z = (z - z_mean) / z_std
    z_test = (z_test - z_mean) / z_std

    # --------------------------------------------------------
    # True ITE should be computed without outcome noise.
    # --------------------------------------------------------
    t0_np = np.zeros_like(true_u_test.unsqueeze(1).cpu().numpy())
    t1_np = np.ones_like(true_u_test.unsqueeze(1).cpu().numpy())

    y0_true = call_outcome_func(
        outcome_func=outcome_func,
        u_np=true_u_test.unsqueeze(1).cpu().numpy(),
        t_np=t0_np,
        z_np=z_test.cpu().numpy(),
        ua0_np=ua0_test_np,
        ua1_np=ua1_test_np,
        add_noise=False,
    )

    y1_true = call_outcome_func(
        outcome_func=outcome_func,
        u_np=true_u_test.unsqueeze(1).cpu().numpy(),
        t_np=t1_np,
        z_np=z_test.cpu().numpy(),
        ua0_np=ua0_test_np,
        ua1_np=ua1_test_np,
        add_noise=False,
    )

    true_ite_np = (y1_true - y0_true).reshape(-1)

    loader = DataLoader(
        TensorDataset(z, t, y, u_true),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    use_aux = get_use_aux(CHOSEN_VERSION)
    z_y_dim = get_z_y_dim(CHOSEN_VERSION, z.shape[1])

    for config in ablation_configs:
        print("\n" + "-" * 80)
        print(f"Training: {config['name']}")
        print("-" * 80)

        set_seed(10000 + iteration_id)

        start_time = time.time()

        inducing_points = None

        if config["prior_type"] == "sparse_gp":
            M = config["inducing_M"]
            M_eff = min(M, z.shape[0])
            idx = torch.randperm(z.shape[0])[:M_eff]
            inducing_points = z[idx].clone()

        model = CausalGPVAEwithNoise(
            input_dim=z.shape[1],
            hidden_dim=INTERVALGPVAE_HIDDEN_DIM,
            latent_dim=INTERVALGPVAE_LATENT_DIM,
            z_y_dim=z_y_dim,
            use_auxiliary_latents=use_aux,
            gp_lengthscale=GP_LENGTHSCALE,
            gp_variance=GP_VARIANCE,
            gp_noise=GP_NOISE,
            prior_type=config["prior_type"],
            prior_weight=config["prior_weight"],
            std_penalty_weight=10.0,
            inducing_points=inducing_points,
        ).to(DEVICE)

        model = train_intervalgpvae_ablation_model(
            model=model,
            loader=loader,
            device=DEVICE,
            joint_epochs=JOINT_EPOCHS,
            head_epochs=HEAD_EPOCHS,
            vae_refine_epochs=VAE_REFINE_EPOCHS,
            lr_joint=LR_JOINT,
            lr_head=LR_HEAD,
            lr_vae_refine=LR_VAE_REFINE,
        )

        metrics = evaluate_ablation_model(
            model=model,
            z_train=z,
            z_test=z_test,
            true_u_test=true_u_test,
            true_ite_np=true_ite_np,
            ci=ITE_CI,
            n_mc=N_MC,
            device=DEVICE,
        )

        runtime_sec = time.time() - start_time
        metrics["runtime_sec"] = runtime_sec

        row = {
            "combination_id": iteration_id,
            "proxy_id": proxy_id,
            "treatment_id": treat_id,
            "outcome_id": outcome_id,
            "model": config["name"],
            "prior_type": config["prior_type"],
            "inducing_M": config["inducing_M"],
            "chosen_version": CHOSEN_VERSION,
            "N_train": N_TRAIN,
            "N_test": N_TEST,
            "latent_dim": INTERVALGPVAE_LATENT_DIM,
            "hidden_dim": INTERVALGPVAE_HIDDEN_DIM,
            "causal_head_hidden_dim": CAUSAL_HEAD_HIDDEN_DIM,
            "gp_lengthscale": GP_LENGTHSCALE,
            "gp_variance": GP_VARIANCE,
            "gp_noise": GP_NOISE,
            "batch_size": BATCH_SIZE,
            "joint_epochs": JOINT_EPOCHS,
            "head_epochs": HEAD_EPOCHS,
            "vae_refine_epochs": VAE_REFINE_EPOCHS,
            "lr_joint": LR_JOINT,
            "lr_head": LR_HEAD,
            "lr_vae_refine": LR_VAE_REFINE,
            "weight_decay": WEIGHT_DECAY,
            **metrics,
        }

        all_ablation_results.append(row)

        print(f"Finished {config['name']} in {runtime_sec:.2f} seconds")
        print(
            f"PEHE={metrics['PEHE']:.4f}, "
            f"ATE_error={metrics['ATE_error']:.4f}, "
            f"coverage={metrics['coverage']:.4f}, "
            f"width={metrics['interval_width']:.4f}, "
            f"cal_error={metrics['calibration_error']:.4f}, "
            f"latent_abs_corr={metrics['latent_abs_corr']:.4f}, "
            f"raw_latent_abs_corr={metrics['raw_latent_abs_corr']:.4f}, "
            f"avg_smooth_u_std={metrics['avg_smooth_u_std']:.4f}, "
            f"coherence={metrics['latent_distance_coherence']:.4f}, "
            f"kNN={metrics['knn_latent_consistency']:.4f}"
        )

    # Save intermediate results after each combination.
    results_df_tmp = pd.DataFrame(all_ablation_results)

    tmp_path = os.path.join(
        OUTPUT_DIR,
        "ablation_A_intervalgpvae_consistent_partial.csv",
    )

    results_df_tmp.to_csv(tmp_path, index=False)

    print(f"Partial results saved to: {tmp_path}")


# ============================================================
# 11. Save full results
# ============================================================
results_df = pd.DataFrame(all_ablation_results)

results_path = os.path.join(
    OUTPUT_DIR,
    "ablation_A_intervalgpvae_consistent_results.csv",
)

results_df.to_csv(results_path, index=False)

print("\n" + "=" * 100)
print(f"Full results saved to: {results_path}")
print("=" * 100)

print(results_df.head())


# ============================================================
# 12. Summary table
# ============================================================
summary_df = (
    results_df
    .groupby("model")
    .agg({
        "PEHE": ["mean", "std"],
        "ATE_error": ["mean", "std"],
        "coverage": ["mean", "std"],
        "interval_width": ["mean", "std"],
        "calibration_error": ["mean", "std"],
        "NLL": ["mean", "std"],
        "interval_score": ["mean", "std"],
        "latent_RMSE": ["mean", "std"],
        "latent_corr": ["mean", "std"],
        "latent_abs_corr": ["mean", "std"],
        "raw_latent_RMSE": ["mean", "std"],
        "raw_latent_corr": ["mean", "std"],
        "raw_latent_abs_corr": ["mean", "std"],
        "avg_smooth_u_std": ["mean", "std"],
        "latent_distance_coherence": ["mean", "std"],
        "knn_latent_consistency": ["mean", "std"],
        "runtime_sec": ["mean", "std"],
    })
)

summary_path = os.path.join(
    OUTPUT_DIR,
    "ablation_A_intervalgpvae_consistent_summary.csv",
)

summary_df.to_csv(summary_path)

print("\nSummary:")
print(summary_df)
print(f"\nSummary saved to: {summary_path}")


# ============================================================
# 13. Simple dashboard plot
# ============================================================
if len(results_df) > 0:
    plot_df = (
        results_df
        .groupby("model")
        .agg({
            "PEHE": "mean",
            "ATE_error": "mean",
            "coverage": "mean",
            "interval_width": "mean",
            "latent_abs_corr": "mean",
            "runtime_sec": "mean",
        })
        .reset_index()
    )

    x = np.arange(len(plot_df))

    fig, axs = plt.subplots(6, 1, figsize=(14, 24), sharex=True)

    axs[0].bar(x, plot_df["PEHE"].values)
    axs[0].set_ylabel("PEHE")
    axs[0].set_title("Average PEHE by Ablation Model")
    axs[0].grid(True, linestyle="--", alpha=0.5)

    axs[1].bar(x, plot_df["ATE_error"].values)
    axs[1].set_ylabel("ATE Error")
    axs[1].set_title("Average ATE Error by Ablation Model")
    axs[1].grid(True, linestyle="--", alpha=0.5)

    axs[2].bar(x, plot_df["coverage"].values)
    axs[2].axhline(0.90, linestyle="--", color="black", alpha=0.7)
    axs[2].set_ylabel("Coverage")
    axs[2].set_title("Average 90% ITE Interval Coverage by Ablation Model")
    axs[2].grid(True, linestyle="--", alpha=0.5)

    axs[3].bar(x, plot_df["interval_width"].values)
    axs[3].set_ylabel("Interval Width")
    axs[3].set_title("Average ITE Interval Width by Ablation Model")
    axs[3].grid(True, linestyle="--", alpha=0.5)

    axs[4].bar(x, plot_df["latent_abs_corr"].values)
    axs[4].set_ylabel("|Latent Corr|")
    axs[4].set_title("Average Latent Absolute Correlation by Ablation Model")
    axs[4].grid(True, linestyle="--", alpha=0.5)

    axs[5].bar(x, plot_df["runtime_sec"].values)
    axs[5].set_ylabel("Runtime seconds")
    axs[5].set_title("Average Runtime by Ablation Model")
    axs[5].grid(True, linestyle="--", alpha=0.5)

    axs[5].set_xticks(x)
    axs[5].set_xticklabels(plot_df["model"].values, rotation=45, ha="right")

    plt.tight_layout()

    dashboard_path = os.path.join(
        OUTPUT_DIR,
        "ablation_A_intervalgpvae_consistent_dashboard.png",
    )

    plt.savefig(dashboard_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Dashboard saved to: {dashboard_path}")

# Plot results

In [ ]:
##########################################################
##########################################################
# ============================================================
# 11. Revised Plotting Code:
#     Compatible with generated ablation results
#     Supports both A2 model names
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 11.0 Basic checks and plot directory
# ============================================================

if "results_df" not in globals():
    raise NameError(
        "results_df is not defined. Please run the ablation experiment code first."
    )

if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = "ablation_A_intervalgpvae_consistent_outputs"

PLOT_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(PLOT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Handle CI / ITE_CI compatibility
# ------------------------------------------------------------
if "CI" not in globals():
    if "ITE_CI" in globals():
        CI = ITE_CI
    else:
        CI = 0.90

print(f"Using nominal CI = {CI}")


# ============================================================
# 11.1 Standardize model names for plotting
# ============================================================
# The generated code may use either:
#   A2_current_interval_GP
# or:
#   A2_current_interval_GP_consistent_with_second_code
#
# For plotting, we standardize both to:
#   A2_current_interval_GP
# ============================================================

MODEL_NAME_CANONICAL_MAP = {
    "A2_current_interval_GP_consistent_with_second_code": "A2_current_interval_GP",
}

results_df = results_df.copy()

results_df["model"] = results_df["model"].replace(MODEL_NAME_CANONICAL_MAP)


# ============================================================
# 11.2 Short labels and fixed colors
# ============================================================

MODEL_LABEL_MAP = {
    "A0_VAE_standard": "A0",
    "A1_input_dependent_Gaussian": "A1",
    "A2_current_interval_GP": "A2",
    "A3_joint_minibatch_GP_NLL": "A3-NLL",
    "A3_joint_minibatch_GP_KL": "A3-KL",
    "A4_sparse_GP_M32": "A4-32",
    "A4_sparse_GP_M64": "A4-64",
    "A4_sparse_GP_M128": "A4-128",
}

MODEL_ORDER = [
    "A0_VAE_standard",
    "A1_input_dependent_Gaussian",
    "A2_current_interval_GP",
    "A3_joint_minibatch_GP_NLL",
    "A3_joint_minibatch_GP_KL",
    "A4_sparse_GP_M32",
    "A4_sparse_GP_M64",
    "A4_sparse_GP_M128",
]

SHORT_MODEL_ORDER = [MODEL_LABEL_MAP[m] for m in MODEL_ORDER]

MODEL_COLOR_MAP = {
    "A0_VAE_standard": "tab:blue",
    "A1_input_dependent_Gaussian": "tab:orange",
    "A2_current_interval_GP": "tab:green",
    "A3_joint_minibatch_GP_NLL": "tab:red",
    "A3_joint_minibatch_GP_KL": "tab:purple",
    "A4_sparse_GP_M32": "tab:brown",
    "A4_sparse_GP_M64": "tab:pink",
    "A4_sparse_GP_M128": "tab:gray",
}

SHORT_COLOR_MAP = {
    MODEL_LABEL_MAP[m]: MODEL_COLOR_MAP[m] for m in MODEL_ORDER
}

results_df["model_short"] = results_df["model"].map(MODEL_LABEL_MAP)


# ============================================================
# 11.3 Check missing models and missing metrics
# ============================================================

missing_models = [
    m for m in MODEL_ORDER
    if m not in set(results_df["model"].dropna().unique())
]

if len(missing_models) > 0:
    print("\nWarning: The following models are not found in results_df:")
    for m in missing_models:
        print(f"  - {m}")

unknown_models = sorted(
    set(results_df["model"].dropna().unique()) - set(MODEL_ORDER)
)

if len(unknown_models) > 0:
    print("\nWarning: The following models are in results_df but not in MODEL_ORDER:")
    for m in unknown_models:
        print(f"  - {m}")

# Save label mapping for reference
label_mapping_df = pd.DataFrame({
    "short_label": SHORT_MODEL_ORDER,
    "full_model_name": MODEL_ORDER,
    "color": [MODEL_COLOR_MAP[m] for m in MODEL_ORDER],
})

label_mapping_path = os.path.join(PLOT_DIR, "model_label_mapping.csv")
label_mapping_df.to_csv(label_mapping_path, index=False)

print(f"\nSaved model label mapping: {label_mapping_path}")


# ============================================================
# 11.4 Helper functions
# ============================================================

def add_model_mapping_text(fig, x=0.99, y=0.5, fontsize=8):
    """
    Add short-label to full-name mapping text on the right side of a figure.
    """
    mapping_text = "\n".join(
        [f"{MODEL_LABEL_MAP[m]} = {m}" for m in MODEL_ORDER]
    )

    fig.text(
        x,
        y,
        mapping_text,
        ha="right",
        va="center",
        fontsize=fontsize,
        bbox=dict(
            boxstyle="round,pad=0.4",
            facecolor="white",
            edgecolor="gray",
            alpha=0.85,
        ),
    )


def add_color_legend(ax, fontsize=8, ncol=4):
    """
    Add legend using short model labels and fixed colors.
    """
    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="s",
            linestyle="",
            markersize=8,
            markerfacecolor=MODEL_COLOR_MAP[m],
            markeredgecolor=MODEL_COLOR_MAP[m],
            label=MODEL_LABEL_MAP[m],
        )
        for m in MODEL_ORDER
    ]

    ax.legend(
        handles=handles,
        title="Model",
        fontsize=fontsize,
        title_fontsize=fontsize,
        ncol=ncol,
        frameon=True,
    )


def grouped_mean_std(df, metric):
    """
    Compute mean and std by full model name, then attach short labels.
    """
    if metric not in df.columns:
        raise KeyError(
            f"Metric '{metric}' is not found in results_df columns."
        )

    temp = (
        df.groupby("model")[metric]
        .agg(["mean", "std"])
        .reindex(MODEL_ORDER)
        .reset_index()
    )

    temp["model_short"] = temp["model"].map(MODEL_LABEL_MAP)
    temp["color"] = temp["model"].map(MODEL_COLOR_MAP)

    return temp


def save_current_figure(path):
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {path}")


# ============================================================
# 11.5 Metrics to plot
# ============================================================

candidate_plot_metrics = [
    "PEHE",
    "ATE_error",
    "coverage",
    "interval_width",
    "calibration_error",
    "NLL",
    "interval_score",
    "latent_RMSE",
    "latent_abs_corr",
    "latent_distance_coherence",
    "knn_latent_consistency",
    "runtime_sec",
]

plot_metrics = [
    m for m in candidate_plot_metrics
    if m in results_df.columns
]

missing_metrics = [
    m for m in candidate_plot_metrics
    if m not in results_df.columns
]

if len(missing_metrics) > 0:
    print("\nWarning: The following metrics are missing and will be skipped:")
    for m in missing_metrics:
        print(f"  - {m}")

print("\nMetrics to plot:")
for m in plot_metrics:
    print(f"  - {m}")


# ============================================================
# 11.6 Save one colored bar plot for each metric
# ============================================================

for metric in plot_metrics:
    temp = grouped_mean_std(results_df, metric)

    fig, ax = plt.subplots(figsize=(12, 5))

    x = np.arange(len(temp))

    ax.bar(
        x,
        temp["mean"].values,
        yerr=temp["std"].values,
        capsize=4,
        color=temp["color"].values,
        edgecolor="black",
        linewidth=0.6,
        alpha=0.85,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(temp["model_short"], fontsize=10)
    ax.set_ylabel(metric)
    ax.set_title(f"Ablation A: {metric}")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    mapping_text = " | ".join(
        [f"{MODEL_LABEL_MAP[m]}: {m}" for m in MODEL_ORDER]
    )

    fig.text(
        0.5,
        -0.05,
        mapping_text,
        ha="center",
        va="top",
        fontsize=8,
        wrap=True,
    )

    plt.tight_layout()

    fig_path = os.path.join(PLOT_DIR, f"ablation_A_{metric}.png")
    save_current_figure(fig_path)


# ============================================================
# 11.7 Save compact colored dashboard figure
# ============================================================

dashboard_metrics = [
    m for m in [
        "PEHE",
        "ATE_error",
        "coverage",
        "interval_width",
        "calibration_error",
        "latent_abs_corr",
        "latent_distance_coherence",
        "runtime_sec",
    ]
    if m in results_df.columns
]

if len(dashboard_metrics) > 0:
    n_panels = len(dashboard_metrics)
    n_cols = 2
    n_rows = int(np.ceil(n_panels / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 5.5 * n_rows),
    )

    axes = np.asarray(axes).reshape(-1)

    for ax, metric in zip(axes, dashboard_metrics):
        temp = grouped_mean_std(results_df, metric)

        x = np.arange(len(temp))

        ax.bar(
            x,
            temp["mean"].values,
            yerr=temp["std"].values,
            capsize=3,
            color=temp["color"].values,
            edgecolor="black",
            linewidth=0.5,
            alpha=0.85,
        )

        ax.set_xticks(x)
        ax.set_xticklabels(temp["model_short"], fontsize=9)
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.grid(axis="y", linestyle="--", alpha=0.5)

    # Remove unused axes
    for j in range(len(dashboard_metrics), len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(
        "Ablation A: Latent Prior / Regularizer Summary",
        fontsize=18,
    )

    add_model_mapping_text(fig, x=0.99, y=0.5, fontsize=8)

    plt.tight_layout(rect=[0, 0, 0.82, 0.96])

    dashboard_path = os.path.join(PLOT_DIR, "ablation_A_dashboard.png")
    save_current_figure(dashboard_path)


# ============================================================
# 11.8 Save all-metrics comprehensive dashboard
# ============================================================

def plot_comprehensive_dashboard(df, metrics, plot_dir):
    if len(metrics) == 0:
        print("No metrics available for comprehensive dashboard.")
        return None

    n_cols = 3
    n_rows = int(np.ceil(len(metrics) / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(22, 7 * n_rows),
    )

    axes = np.asarray(axes).reshape(-1)

    for i, metric in enumerate(metrics):
        ax = axes[i]
        temp = grouped_mean_std(df, metric)
        x = np.arange(len(temp))

        ax.bar(
            x,
            temp["mean"].values,
            yerr=temp["std"].values,
            capsize=3,
            color=temp["color"].values,
            edgecolor="black",
            linewidth=0.5,
            alpha=0.85,
        )

        ax.set_xticks(x)
        ax.set_xticklabels(temp["model_short"], fontsize=12)
        ax.set_ylabel(metric.replace("_", " ").title(), fontsize=12)
        ax.set_title(
            metric.replace("_", " ").title(),
            fontsize=18,
            fontweight="bold",
        )
        ax.grid(axis="y", linestyle="--", alpha=0.5)

    for j in range(len(metrics), len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle(
        "Ablation A: Latent Prior / Regularizer Comprehensive Metrics",
        fontsize=28,
        y=0.98,
    )

    mapping_text = "\n".join([
        "  ".join([f"{MODEL_LABEL_MAP[m]}: {m}" for m in MODEL_ORDER[:4]]),
        "  ".join([f"{MODEL_LABEL_MAP[m]}: {m}" for m in MODEL_ORDER[4:]]),
    ])

    fig.text(
        0.5,
        0.02,
        mapping_text,
        ha="center",
        va="bottom",
        fontsize=18,
        bbox=dict(
            boxstyle="round,pad=0.8",
            facecolor="whitesmoke",
            edgecolor="gray",
        ),
    )

    plt.tight_layout(rect=[0, 0.09, 1, 0.96])

    save_path = os.path.join(
        plot_dir,
        "ablation_A_full_dashboard_final.png",
    )

    save_current_figure(save_path)

    return save_path


plot_comprehensive_dashboard(results_df, plot_metrics, PLOT_DIR)


# ============================================================
# 11.9 PEHE vs Coverage trade-off
# ============================================================

required_tradeoff_metrics = [
    "PEHE",
    "coverage",
    "interval_width",
    "latent_abs_corr",
    "latent_distance_coherence",
]

available_for_tradeoff = all(
    m in results_df.columns for m in required_tradeoff_metrics
)

if available_for_tradeoff:
    tradeoff_df = (
        results_df
        .groupby("model")
        .agg(
            PEHE_mean=("PEHE", "mean"),
            PEHE_std=("PEHE", "std"),
            coverage_mean=("coverage", "mean"),
            coverage_std=("coverage", "std"),
            width_mean=("interval_width", "mean"),
            latent_abs_corr_mean=("latent_abs_corr", "mean"),
            latent_coherence_mean=("latent_distance_coherence", "mean"),
        )
        .reindex(MODEL_ORDER)
        .reset_index()
    )

    tradeoff_df["model_short"] = tradeoff_df["model"].map(MODEL_LABEL_MAP)
    tradeoff_df["color"] = tradeoff_df["model"].map(MODEL_COLOR_MAP)

    # --------------------------------------------------------
    # PEHE vs Coverage
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))

    for _, row in tradeoff_df.iterrows():
        if pd.isna(row["PEHE_mean"]) or pd.isna(row["coverage_mean"]):
            continue

        ax.scatter(
            row["PEHE_mean"],
            row["coverage_mean"],
            s=120,
            color=row["color"],
            edgecolor="black",
            linewidth=0.6,
            alpha=0.9,
        )

        ax.text(
            row["PEHE_mean"],
            row["coverage_mean"],
            row["model_short"],
            fontsize=10,
            ha="left",
            va="bottom",
        )

    ax.axhline(
        CI,
        linestyle="--",
        alpha=0.7,
        label=f"Nominal coverage = {CI}",
    )

    ax.set_xlabel("Mean PEHE ↓")
    ax.set_ylabel("Mean coverage")
    ax.set_title("Ablation A: PEHE vs Coverage Trade-off")
    ax.grid(True, linestyle="--", alpha=0.5)

    add_color_legend(ax, fontsize=8, ncol=4)
    add_model_mapping_text(fig, x=0.99, y=0.5, fontsize=8)

    plt.tight_layout(rect=[0, 0, 0.78, 1])

    tradeoff_path = os.path.join(
        PLOT_DIR,
        "ablation_A_PEHE_vs_coverage.png",
    )

    save_current_figure(tradeoff_path)

    # --------------------------------------------------------
    # Interval width vs Coverage
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))

    for _, row in tradeoff_df.iterrows():
        if pd.isna(row["width_mean"]) or pd.isna(row["coverage_mean"]):
            continue

        ax.scatter(
            row["width_mean"],
            row["coverage_mean"],
            s=120,
            color=row["color"],
            edgecolor="black",
            linewidth=0.6,
            alpha=0.9,
        )

        ax.text(
            row["width_mean"],
            row["coverage_mean"],
            row["model_short"],
            fontsize=10,
            ha="left",
            va="bottom",
        )

    ax.axhline(
        CI,
        linestyle="--",
        alpha=0.7,
        label=f"Nominal coverage = {CI}",
    )

    ax.set_xlabel("Mean interval width ↓")
    ax.set_ylabel("Mean coverage")
    ax.set_title("Ablation A: Interval Width vs Coverage")
    ax.grid(True, linestyle="--", alpha=0.5)

    add_color_legend(ax, fontsize=8, ncol=4)
    add_model_mapping_text(fig, x=0.99, y=0.5, fontsize=8)

    plt.tight_layout(rect=[0, 0, 0.78, 1])

    width_coverage_path = os.path.join(
        PLOT_DIR,
        "ablation_A_width_vs_coverage.png",
    )

    save_current_figure(width_coverage_path)

    # --------------------------------------------------------
    # Latent coherence vs PEHE
    # --------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))

    for _, row in tradeoff_df.iterrows():
        if pd.isna(row["latent_coherence_mean"]) or pd.isna(row["PEHE_mean"]):
            continue

        ax.scatter(
            row["latent_coherence_mean"],
            row["PEHE_mean"],
            s=120,
            color=row["color"],
            edgecolor="black",
            linewidth=0.6,
            alpha=0.9,
        )

        ax.text(
            row["latent_coherence_mean"],
            row["PEHE_mean"],
            row["model_short"],
            fontsize=10,
            ha="left",
            va="bottom",
        )

    ax.set_xlabel("Mean latent distance coherence ↑")
    ax.set_ylabel("Mean PEHE ↓")
    ax.set_title("Ablation A: Latent Coherence vs PEHE")
    ax.grid(True, linestyle="--", alpha=0.5)

    add_color_legend(ax, fontsize=8, ncol=4)
    add_model_mapping_text(fig, x=0.99, y=0.5, fontsize=8)

    plt.tight_layout(rect=[0, 0, 0.78, 1])

    coherence_pehe_path = os.path.join(
        PLOT_DIR,
        "ablation_A_latent_coherence_vs_PEHE.png",
    )

    save_current_figure(coherence_pehe_path)

else:
    print(
        "\nSkipping trade-off plots because some required metrics are missing:"
    )
    for m in required_tradeoff_metrics:
        if m not in results_df.columns:
            print(f"  - {m}")


# ============================================================
# 11.10 PEHE across combinations
# ============================================================

if "combination_id" in results_df.columns and "PEHE" in results_df.columns:
    fig, ax = plt.subplots(figsize=(14, 6))

    for model_name in MODEL_ORDER:
        sub_df = results_df[results_df["model"] == model_name].sort_values(
            "combination_id"
        )

        if len(sub_df) == 0:
            continue

        ax.plot(
            sub_df["combination_id"],
            sub_df["PEHE"],
            marker="o",
            linewidth=2,
            markersize=5,
            color=MODEL_COLOR_MAP[model_name],
            label=MODEL_LABEL_MAP[model_name],
        )

    ax.set_xlabel("Combination ID")
    ax.set_ylabel("PEHE ↓")
    ax.set_title("Ablation A: PEHE Across Synthetic Combinations")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(title="Model", fontsize=9, ncol=4)

    mapping_text = " | ".join(
        [f"{MODEL_LABEL_MAP[m]}: {m}" for m in MODEL_ORDER]
    )

    fig.text(
        0.5,
        -0.03,
        mapping_text,
        ha="center",
        va="top",
        fontsize=8,
        wrap=True,
    )

    plt.tight_layout()

    pehe_combo_path = os.path.join(
        PLOT_DIR,
        "ablation_A_PEHE_across_combinations.png",
    )

    save_current_figure(pehe_combo_path)


# ============================================================
# 11.11 Coverage across combinations
# ============================================================

if "combination_id" in results_df.columns and "coverage" in results_df.columns:
    fig, ax = plt.subplots(figsize=(14, 6))

    for model_name in MODEL_ORDER:
        sub_df = results_df[results_df["model"] == model_name].sort_values(
            "combination_id"
        )

        if len(sub_df) == 0:
            continue

        ax.plot(
            sub_df["combination_id"],
            sub_df["coverage"],
            marker="o",
            linewidth=2,
            markersize=5,
            color=MODEL_COLOR_MAP[model_name],
            label=MODEL_LABEL_MAP[model_name],
        )

    ax.axhline(
        CI,
        linestyle="--",
        alpha=0.7,
        label=f"Nominal coverage = {CI}",
    )

    ax.set_xlabel("Combination ID")
    ax.set_ylabel("Coverage")
    ax.set_title("Ablation A: Coverage Across Synthetic Combinations")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(title="Model", fontsize=9, ncol=4)

    mapping_text = " | ".join(
        [f"{MODEL_LABEL_MAP[m]}: {m}" for m in MODEL_ORDER]
    )

    fig.text(
        0.5,
        -0.03,
        mapping_text,
        ha="center",
        va="top",
        fontsize=8,
        wrap=True,
    )

    plt.tight_layout()

    coverage_combo_path = os.path.join(
        PLOT_DIR,
        "ablation_A_coverage_across_combinations.png",
    )

    save_current_figure(coverage_combo_path)


# ============================================================
# 11.12 Metric correlation heatmap
# ============================================================

available_metrics = [m for m in plot_metrics if m in results_df.columns]

if len(available_metrics) >= 2:
    corr_matrix = results_df[available_metrics].corr()

    fig, ax = plt.subplots(figsize=(12, 10))

    im = ax.imshow(
        corr_matrix.values,
        aspect="auto",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
    )

    ax.set_xticks(np.arange(len(available_metrics)))
    ax.set_yticks(np.arange(len(available_metrics)))

    ax.set_xticklabels(
        available_metrics,
        rotation=45,
        ha="right",
        fontsize=8,
    )

    ax.set_yticklabels(
        available_metrics,
        fontsize=8,
    )

    for i in range(len(available_metrics)):
        for j in range(len(available_metrics)):
            value = corr_matrix.values[i, j]

            if np.isfinite(value):
                text_value = f"{value:.2f}"
            else:
                text_value = "NA"

            ax.text(
                j,
                i,
                text_value,
                ha="center",
                va="center",
                fontsize=7,
            )

    fig.colorbar(im, ax=ax)
    ax.set_title("Correlation Matrix between Latent and Causal Metrics")

    plt.tight_layout()

    corr_path = os.path.join(
        PLOT_DIR,
        "ablation_A_metric_correlation.png",
    )

    save_current_figure(corr_path)
else:
    print("Skipping correlation heatmap because fewer than two metrics are available.")


# ============================================================
# 11.13 Compact summary table as image
# ============================================================

if available_for_tradeoff:
    compact_table_for_plot = tradeoff_df.copy()

    compact_table_for_plot = compact_table_for_plot[
        [
            "model_short",
            "model",
            "PEHE_mean",
            "coverage_mean",
            "width_mean",
            "latent_abs_corr_mean",
            "latent_coherence_mean",
        ]
    ]

    compact_table_for_plot = compact_table_for_plot.rename(
        columns={
            "model_short": "Label",
            "model": "Full model name",
            "PEHE_mean": "PEHE",
            "coverage_mean": "Coverage",
            "width_mean": "Width",
            "latent_abs_corr_mean": "Latent |corr|",
            "latent_coherence_mean": "Coherence",
        }
    )

    for col in compact_table_for_plot.columns:
        if col not in ["Label", "Full model name"]:
            compact_table_for_plot[col] = compact_table_for_plot[col].map(
                lambda x: f"{x:.4f}" if pd.notna(x) else "NA"
            )

    fig, ax = plt.subplots(figsize=(16, 4.5))
    ax.axis("off")

    table = ax.table(
        cellText=compact_table_for_plot.values,
        colLabels=compact_table_for_plot.columns,
        cellLoc="center",
        loc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.6)

    ax.set_title(
        "Ablation A: Compact Summary Table",
        fontsize=14,
        pad=12,
    )

    plt.tight_layout()

    table_img_path = os.path.join(
        PLOT_DIR,
        "ablation_A_compact_summary_table.png",
    )

    save_current_figure(table_img_path)


# ============================================================
# 12. Compact comparison table for paper/rebuttal
# ============================================================

agg_dict = {}

if "PEHE" in results_df.columns:
    agg_dict["PEHE_mean"] = ("PEHE", "mean")
    agg_dict["PEHE_std"] = ("PEHE", "std")

if "ATE_error" in results_df.columns:
    agg_dict["ATE_error_mean"] = ("ATE_error", "mean")

if "coverage" in results_df.columns:
    agg_dict["coverage_mean"] = ("coverage", "mean")

if "interval_width" in results_df.columns:
    agg_dict["width_mean"] = ("interval_width", "mean")

if "calibration_error" in results_df.columns:
    agg_dict["calibration_error_mean"] = ("calibration_error", "mean")

if "NLL" in results_df.columns:
    agg_dict["NLL_mean"] = ("NLL", "mean")

if "interval_score" in results_df.columns:
    agg_dict["interval_score_mean"] = ("interval_score", "mean")

if "latent_abs_corr" in results_df.columns:
    agg_dict["latent_abs_corr_mean"] = ("latent_abs_corr", "mean")

if "latent_distance_coherence" in results_df.columns:
    agg_dict["latent_coherence_mean"] = ("latent_distance_coherence", "mean")

if "knn_latent_consistency" in results_df.columns:
    agg_dict["knn_consistency_mean"] = ("knn_latent_consistency", "mean")

if "runtime_sec" in results_df.columns:
    agg_dict["runtime_mean"] = ("runtime_sec", "mean")

if len(agg_dict) > 0:
    compact_summary = (
        results_df
        .groupby("model")
        .agg(**agg_dict)
        .reindex(MODEL_ORDER)
        .reset_index()
    )

    compact_summary.insert(
        0,
        "model_short",
        compact_summary["model"].map(MODEL_LABEL_MAP),
    )

    compact_path = os.path.join(
        OUTPUT_DIR,
        "ablation_A_compact_summary.csv",
    )

    compact_summary.to_csv(compact_path, index=False)

    print("\nCompact summary:")
    print(compact_summary)
    print(f"\nCompact summary saved to: {compact_path}")


print("\nAll revised colored figures have been saved to:")
print(PLOT_DIR)

# End